# CSAI-4-IoT — Cumulative Final: Base vs Extreme (Seeds 43–52)

Execução final do estudo **Cumulative / Slow Drift**, derivada do Stress Test V2 validado.

### Objetivo
Testar, com **10 seeds independentes (43–52)**, se a intensificação controlada da deriva aumenta a detectabilidade temporal e/ou o impacto adversarial, preservando a semântica do cenário: **1 atacante persistente**, 100 rounds, IID + Non-IID Dirichlet e as mesmas 9 estratégias do benchmark pós-banca.

### Perfis mantidos
- **Base:** epsilon = 0,20 constante durante a janela ativa; reproduz a configuração de referência.
- **Extreme:** epsilon cresce linearmente de 0,20 até 4,00; stress test superior validado no piloto V2.

Os perfis intermediários FullPoison e Strong foram removidos desta execução final porque o piloto já estabeleceu seu papel intermediário e a comparação científica de interesse é **Base vs Extreme**.

### Métricas prioritárias
- Detection Recall
- Detection Delay
- Honest Rejection Rate
- ASR crítico em S12–S16
- F1 crítico em S12–S16

Seed 42 permanece exclusiva de calibração e não entra na inferência.


## Configuração do piloto V2

- Dataset: `NBaIoT_MixedDevice_9Sources_Balanced`
- Seed: **43** (modo `PILOT`)
- 100 rounds
- 10 clientes federados
- 1 atacante Cumulative persistente
- Partições: IID + NonIID-Dirichlet (alpha=0,5)
- 9 estratégias do benchmark final
- tau = 0,05
- mesmas features, harmonização, trigger, ASR, ablação e Circuit Breaker da campanha final
- breakpoint calculado **somente durante rounds com ataque ativo**

### Saída
`09_RESULTADOS_FINAIS_TESE/07_CUMULATIVE_STRESS_TEST_V2_100R_SEED43_EPS020_100_200_400/`

Além dos arquivos padrão, o notebook gera `cumulative_stress_profile.csv`, `cumulative_stress_breakpoints_by_seed.csv` e `cumulative_stress_summary.csv`.


In [1]:
# ==============================================================================
# UNIFIED FL BENCHMARK PIPELINE — CSAI-4-IoT | PÓS-BANCA 2026
# ==============================================================================
# Executa e padroniza 4 cenários adversariais:
#   1) Cumulative / Cumulativo / Slow Drift
#   2) Scaling / Model Replacement
#   3) Sybil / Identity Flood
#   4) On-Off / Sleeping Cell
#
# Saídas padronizadas por dataset:
#   unified_results/<dataset>/<attack>/<partition_mode>/
#       raw_results.csv
#       summary_by_stage.csv
#       summary_by_strategy.csv
#       final_stage_ranking.csv
#       execution_report.csv
#       plots/F1_over_stage.png
#       plots/ASR_over_stage.png
#       plots/Time_over_stage.png
#
# Saídas consolidadas:
#   unified_results/all_raw_results.csv
#   unified_results/all_summary_by_stage.csv
#   unified_results/all_final_stage_ranking.csv
#   unified_results/all_diagnostic.csv
#   unified_results.zip
#
# Uso no Colab:
#   1. Ajuste ATTACKS_TO_RUN = ["Cumulative"] ou uma lista menor.
#   2. Execute a célula.
#   3. Faça upload dos datasets CSV curados.
# ==============================================================================

import subprocess
import sys
import time
import os
import io
import warnings
import shutil
import zipfile
import unicodedata
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import f1_score
from sklearn.metrics.pairwise import cosine_distances
from scipy.spatial.distance import cdist
from scipy import stats
from itertools import combinations

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

try:
    import hdbscan
except ImportError:
    try:
        from sklearn.cluster import HDBSCAN as _SklearnHDBSCAN
        class _HDBSCANShim:
            HDBSCAN = _SklearnHDBSCAN
        hdbscan = _HDBSCANShim()
        print("Usando sklearn.cluster.HDBSCAN como fallback compatível.")
    except ImportError:
        print("Instalando dependência hdbscan...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "hdbscan", "-q"])
        import hdbscan

warnings.filterwarnings("ignore")

# ==============================================================================
# 1. CONFIGURAÇÕES GERAIS
# ==============================================================================

ATTACKS_TO_RUN = ["Cumulative"]
# Exemplo para rodar apenas um:
# ATTACKS_TO_RUN = ["Scaling"]

CALIBRATION_SEED = 42
VALIDATION_SEED = 43
FINAL_BENCHMARK_SEEDS = list(range(43, 53))
N_FEDERATED_ROUNDS = 100

# Cumulative Final: execução inferencial com Seeds 43..52.
# Seed 42 permanece exclusiva da calibração de tau e é excluída da inferência.
CUMULATIVE_STRESS_MODE = "FINAL"  # execução final: Seeds 43..52
RUN_SEEDS = [VALIDATION_SEED] if CUMULATIVE_STRESS_MODE == "PILOT" else list(FINAL_BENCHMARK_SEEDS)
N_ROUNDS_STATISTICAL = len(RUN_SEEDS)
RUN_VERSION = "CSAI4IoT_Cumulative_Final_BaseVsExtreme_100R_Seeds43_52"

# -------------------------------------------------------------------
# SMOKE TEST / PILOTO PRÉ-BENCHMARK
# -------------------------------------------------------------------
SMOKE_TEST_MODE = False
SMOKE_TEST_SEEDS = 1
SMOKE_TEST_DATASETS = []
SMOKE_TEST_ATTACKS = ["Cumulative", "Scaling", "Sybil", "OnOff"]
SMOKE_TEST_PARTITIONS = ["IID", "NonIID_Dirichlet"]
SMOKE_TEST_EXPECTED_ROUNDS = N_FEDERATED_ROUNDS
SMOKE_TEST_FAIL_FAST = True

# Seed 42 permanece exclusiva da calibração de tau.
# Cross-device usa Seed 43 para manter comparabilidade direta com o benchmark final.
INITIAL_SEED = RUN_SEEDS[0]

# -------------------------------------------------------------------
# COLAB — CROSS-DEVICE N-BaIoT / 9 dispositivos
# -------------------------------------------------------------------
COLAB_DRIVE_MODE = True

PROJECT_ROOT_REL = Path(
    "00_Tese_Benchmark"
) / "01_Benchmark_Pos_Banca_2026"

COLAB_DATASET_DIR_REL = (
    PROJECT_ROOT_REL
    / "02_DADOS_HARMONIZADOS"
    / "04_NBAIOT_DISPOSITIVOS_INDIVIDUAIS_BALANCEADOS"
    / "Seed_42"
)

COLAB_RESULTS_REL = (
    PROJECT_ROOT_REL
    / "09_RESULTADOS_FINAIS_TESE"
    / "08_CUMULATIVE_FINAL_BASE_VS_EXTREME_100R_SEEDS43_52"
)

COLAB_RESULTS_DIRNAME = COLAB_RESULTS_REL.name

EXPECTED_DEVICE_FILES = [
    "NBaIoT_Danmini_Doorbell_Balanced2K.csv",
    "NBaIoT_Ecobee_Thermostat_Balanced2K.csv",
    "NBaIoT_Ennio_Doorbell_Balanced2K.csv",
    "NBaIoT_Philips_B120N10_Baby_Monitor_Balanced2K.csv",
    "NBaIoT_Provision_PT_737E_Security_Camera_Balanced2K.csv",
    "NBaIoT_Provision_PT_838_Security_Camera_Balanced2K.csv",
    "NBaIoT_Samsung_SNH_1011_N_Webcam_Balanced2K.csv",
    "NBaIoT_SimpleHome_XCS7_1002_WHT_Security_Camera_Balanced2K.csv",
    "NBaIoT_SimpleHome_XCS7_1003_WHT_Security_Camera_Balanced2K.csv",
]

# Cenário multifonte / dataset combinado (pós-banca)
ENABLE_COMBINED_DATASET = True
RUN_ONLY_COMBINED_DATASET = True
COMBINED_DATASET_NAME = "NBaIoT_MixedDevice_9Sources_Balanced"
COMBINED_BALANCE_BY_SOURCE = True
MIN_COMMON_FEATURES = 100

# Particionamento federado dos clientes (pós-banca)
# IID é o controle; NonIID_Dirichlet introduz heterogeneidade de rótulos de forma
# reproduzível e parametrizada. Alpha menor => maior heterogeneidade.
CLIENT_PARTITION_MODES = ["IID", "NonIID_Dirichlet"]
NONIID_DIRICHLET_ALPHA = 0.5
NONIID_MAX_ATTEMPTS = 200
MIN_CLIENT_SAMPLES = 10

# Cardinalidade federada principal pós-banca.
# Mantemos N=10 nos quatro cenários; a intensidade adversarial varia pelo ataque,
# não pelo tamanho total da federação.
N_MAIN_CLIENTS = 10
DEFAULT_NODES = [f"Node{i+1}" for i in range(N_MAIN_CLIENTS)]

# Mapeamento semântico opcional para nomes equivalentes entre datasets.
# Acrescente aliases aqui somente quando houver equivalência semântica documentada.
FEATURE_ALIASES = {
    # "src_bytes": "bytes_source",
    # "dst_bytes": "bytes_destination",
}

STRATEGIES = [
    "FedAvg",
    "Krum",
    "FLAME",
    "FLTrust",
    "Adaptive_Ultimate"
]

# Configurações do modelo
HIDDEN_LAYERS_DEFAULT = (64,)
HIDDEN_LAYERS_CUMULATIVE = (32,)
LOCAL_EPOCHS_DEFAULT = 3
LOCAL_EPOCHS_CUMULATIVE = 2

# Parâmetros comuns de ataque
POISON_RATE = 0.5

# Trigger adaptado ao dataset e escolhido SOMENTE a partir do conjunto de treino.
# Evita usar um valor bruto arbitrário (como 5.0) que possa, sozinho, deslocar
# amostras maliciosas para a classe benigna.
TRIGGER_QUANTILE = 0.95
TRIGGER_MAX_CLEAN_ASR = 0.10
TRIGGER_MAX_CLEAN_DELTA = 0.05
TRIGGER_MIN_UNIQUE_VALUES = 10

# Semântica explícita das classes para o IDS:
#   0 = tráfego benigno
#   1 = tráfego malicioso/ataque
BENIGN_CLASS = 0
MALICIOUS_CLASS = 1

# Backdoor de evasão:
# o atacante parte de amostras maliciosas e tenta fazê-las ser classificadas
# como benignas após a aplicação do trigger.
BACKDOOR_SOURCE_CLASS = MALICIOUS_CLASS
BACKDOOR_TARGET_CLASS = BENIGN_CLASS

# Parâmetros específicos
SCALING_FACTOR = 100.0
# Intensidade relativa da deriva cumulativa.
# Implementação alinhada à Eq. (2.7) da tese:
# delta_mal = delta_honest + epsilon * grad_drift_t
# onde grad_drift_t é aproximado pela diferença entre a atualização obtida
# com dados localmente envenenados e a atualização limpa do mesmo cliente.
DRIFT_EPSILON = 0.20
CUMULATIVE_ATTACK_START_STAGE = 1

# -------------------------------------------------------------------
# CUMULATIVE STRESS TEST — SENSIBILIDADE DA DERIVA
# -------------------------------------------------------------------
# Base reproduz EXATAMENTE o benchmark final: epsilon=0.20 constante
# durante todos os rounds ativos. FullPoison, Strong e Extreme preservam o mesmo início
# (epsilon=0.20) e aumentam gradualmente a intensidade até 1.00, 2.00 e 4.00
# no round final. Assim, altera-se somente a magnitude temporal da deriva;
# permanece 1 atacante persistente, sem transformar o cenário em Scaling.
CUMULATIVE_STRESS_PROFILES = {
    "Base":       {"max_epsilon": 0.20, "description": "controle idêntico ao benchmark final"},
    "FullPoison": {"max_epsilon": 1.00, "description": "deriva progressiva até a atualização local envenenada completa"},
    "Strong":     {"max_epsilon": 2.00, "description": "extrapolação 2x da direção específica de deriva adversarial"},
    "Extreme":    {"max_epsilon": 4.00, "description": "stress test 4x da direção específica de deriva adversarial"},
}
CUMULATIVE_PROFILES_TO_RUN = ["Base", "Extreme"]


# Compatibilidade temporal com o benchmark original (17 estágios: 0..16).
# Os 100 rounds atuais são projetados proporcionalmente sobre essa escala,
# preservando os mesmos pontos relativos de ativação dos ataques.
LEGACY_MAX_STAGE = 16

def round_to_legacy_stage(round_idx, total_rounds=N_FEDERATED_ROUNDS):
    if total_rounds <= 1:
        return 0
    return int(np.floor(round_idx * LEGACY_MAX_STAGE / (total_rounds - 1)))


def cumulative_epsilon_for_round(round_idx, profile_name):
    """Retorna epsilon_t para o Cumulative Stress Test.

    Base é matematicamente idêntico ao benchmark final (epsilon=0.20 fixo).
    FullPoison/Strong/Extreme começam em 0.20 no primeiro round ativo e crescem linearmente
    até 1.00/2.00/4.00 no round 99.
    """
    if profile_name not in CUMULATIVE_STRESS_PROFILES:
        raise ValueError(f"Perfil Cumulative desconhecido: {profile_name}")

    legacy_stage = round_to_legacy_stage(round_idx)
    if legacy_stage < CUMULATIVE_ATTACK_START_STAGE:
        return 0.0

    active_rounds = [
        r for r in range(N_FEDERATED_ROUNDS)
        if round_to_legacy_stage(r) >= CUMULATIVE_ATTACK_START_STAGE
    ]
    start_round = min(active_rounds)
    end_round = N_FEDERATED_ROUNDS - 1
    progress = (round_idx - start_round) / max(1, end_round - start_round)
    progress = float(np.clip(progress, 0.0, 1.0))

    max_epsilon = float(CUMULATIVE_STRESS_PROFILES[profile_name]["max_epsilon"])
    return float(DRIFT_EPSILON + (max_epsilon - DRIFT_EPSILON) * progress)


def validate_cumulative_stress_profiles():
    rows = []
    for profile in CUMULATIVE_PROFILES_TO_RUN:
        for r in range(N_FEDERATED_ROUNDS):
            rows.append({
                "Profile": profile,
                "Round": r,
                "Legacy_Stage": round_to_legacy_stage(r),
                "Attack_Active": int(round_to_legacy_stage(r) >= CUMULATIVE_ATTACK_START_STAGE),
                "Epsilon": cumulative_epsilon_for_round(r, profile),
                "Persistent_Attackers": int(round_to_legacy_stage(r) >= CUMULATIVE_ATTACK_START_STAGE),
            })
    df = pd.DataFrame(rows)

    # Fail-fast científico: Base deve reproduzir epsilon=0.20 em TODO round ativo.
    base_active = df[(df["Profile"] == "Base") & (df["Attack_Active"] == 1)]
    if not np.allclose(base_active["Epsilon"].values, DRIFT_EPSILON):
        raise AssertionError("Perfil Base deixou de reproduzir o benchmark final.")

    expected_final = {"Base": 0.20, "Extreme": 4.00}
    for profile, expected in expected_final.items():
        observed = float(df[(df["Profile"] == profile) & (df["Round"] == 99)]["Epsilon"].iloc[0])
        if not np.isclose(observed, expected):
            raise AssertionError(f"{profile}: epsilon final {observed} != {expected}")
    return df

# -------------------------------------------------------------------
# PERFIL DE PRESSÃO ADVERSARIAL ALINHADO À TESE
# -------------------------------------------------------------------
# A tese especifica progressão por estágios S0..S16 e uma janela crítica
# S12..S16 com maioria maliciosa sustentada. Os percentuais intermediários
# abaixo são uma operacionalização explícita e reproduzível dessa progressão
# para N=10 clientes: 10% -> 30% -> 50% -> 70%.
# O estágio S0 é sempre limpo (warm-up).
THESIS_PRESSURE_BANDS = [
    (0, 0, 0.00),
    (1, 4, 0.10),
    (5, 8, 0.30),
    (9, 11, 0.50),
    (12, 16, 0.70),
]

def pressure_fraction_for_legacy_stage(legacy_stage):
    for start, end, frac in THESIS_PRESSURE_BANDS:
        if start <= legacy_stage <= end:
            return float(frac)
    return 0.0

def pressure_attackers_for_legacy_stage(legacy_stage, n_nodes):
    frac = pressure_fraction_for_legacy_stage(legacy_stage)
    if frac <= 0:
        return 0
    return min(n_nodes, max(1, int(round(frac * n_nodes))))

# On-Off: clientes comprometidos seguem a mesma progressão de pressão,
# porém só injetam poisoning nas janelas ON. As janelas preservam o caráter
# intermitente/sleeping-cell e ficam mais longas na fase final.
THESIS_ONOFF_ACTIVE_STAGES = {1, 3, 5, 6, 9, 10, 12, 13, 15, 16}

def validate_thesis_pressure_profile(n_nodes=N_MAIN_CLIENTS):
    rows = []
    for st in range(17):
        frac = pressure_fraction_for_legacy_stage(st)
        n_att = pressure_attackers_for_legacy_stage(st, n_nodes)
        rows.append({
            "Legacy_Stage": st,
            "Pressure_Fraction": frac,
            "Compromised_Clients": n_att,
            "OnOff_Active": int(st in THESIS_ONOFF_ACTIVE_STAGES),
        })
    df = pd.DataFrame(rows)
    # Fail-fast: S12..S16 deve representar maioria maliciosa (>50%).
    critical = df[df["Legacy_Stage"].between(12, 16)]
    if not (critical["Compromised_Clients"] > (n_nodes / 2)).all():
        raise AssertionError("Perfil inválido: S12-S16 deve manter maioria maliciosa.")
    return df

# Parâmetros das defesas
ALPHA_PENALTY = 0.85
BETA_RECOVERY = 0.10
TRUST_THRESHOLD = 0.05  # calibrado na Seed 42; seeds 43-52 não participaram do tuning
FLAME_NOISE_SIGMA = 0.001

# Circuit Breaker adaptativo — modo experimental pós-banca.
# A maioria absoluta permanece como estado NORMAL, mas a queda abaixo da maioria
# não congela imediatamente a rodada. O sistema entra em modo DEGRADADO quando
# ainda existe suporte confiável suficiente e reduz progressivamente a atualização.
# O BREAK é reservado para perda persistente de suporte confiável.
CB_CALIBRATION_ROUNDS = 7  # rounds 0..6; precedem o primeiro ataque nos cenários atuais
CB_MIN_TRUSTED_FRACTION = 0.30
CB_MIN_EFFECTIVE_FRACTION = 0.20
CB_MIN_LAMBDA = 0.25
CB_ALIGNMENT_THRESHOLD = 0.0
CB_TRIGGER_PATIENCE = 2
CB_RELEASE_PATIENCE = 2

# Ablação / modularização da defesa (pós-banca)
# IMPORTANTE: a implementação atual possui um único estado temporal (trust_memory).
# Não há uma variável de reputação independente; portanto, memória/reputação são tratadas
# como um único mecanismo temporal para evitar uma ablação artificial.
DEFENSE_CONFIG = {
    "use_temporal_memory": True,
    "use_adaptive_weighting": True,
    "use_circuit_breaker": True,
    "circuit_breaker_mode": "adaptive",
}

ABLATION_CONFIGS = {
    # Modelo completo final: CB adaptativo pós-calibração.
    "Full": {
        "use_temporal_memory": True,
        "use_adaptive_weighting": True,
        "use_circuit_breaker": True,
        "circuit_breaker_mode": "adaptive",
    },
    # Controle legado: regra de maioria fixa usada antes da revisão pós-banca.
    "FixedCircuitBreaker": {
        "use_temporal_memory": True,
        "use_adaptive_weighting": True,
        "use_circuit_breaker": True,
        "circuit_breaker_mode": "fixed",
    },
    "NoTemporalMemory": {
        "use_temporal_memory": False,
        "use_adaptive_weighting": True,
        "use_circuit_breaker": True,
        "circuit_breaker_mode": "adaptive",
    },
    "NoAdaptiveWeighting": {
        "use_temporal_memory": True,
        "use_adaptive_weighting": False,
        "use_circuit_breaker": True,
        "circuit_breaker_mode": "adaptive",
    },
    "NoCircuitBreaker": {
        "use_temporal_memory": True,
        "use_adaptive_weighting": True,
        "use_circuit_breaker": False,
        "circuit_breaker_mode": "disabled",
    },
}

# O Full representa a versão final do CSAI-4-IoT com CB adaptativo.
# As ablações preservam essa base e alteram apenas um componente por vez.

# Executor efetivo da ablação.
# "Strategy" é o rótulo persistido nos resultados; "base_strategy" define o
# algoritmo executado internamente. Todas as variantes Adaptive reutilizam
# exatamente a mesma implementação de Adaptive_Ultimate, alterando somente
# a configuração do componente removido.
ADAPTIVE_VARIANT_TO_CONFIG = {
    "Adaptive_Ultimate": ("Full", ABLATION_CONFIGS["Full"]),
    "Adaptive_FixedCircuitBreaker": ("FixedCircuitBreaker", ABLATION_CONFIGS["FixedCircuitBreaker"]),
    "Adaptive_NoTemporalMemory": ("NoTemporalMemory", ABLATION_CONFIGS["NoTemporalMemory"]),
    "Adaptive_NoAdaptiveWeighting": ("NoAdaptiveWeighting", ABLATION_CONFIGS["NoAdaptiveWeighting"]),
    "Adaptive_NoCircuitBreaker": ("NoCircuitBreaker", ABLATION_CONFIGS["NoCircuitBreaker"]),
}

STRATEGIES = [
    "FedAvg",
    "Krum",
    "FLAME",
    "FLTrust",
    "Adaptive_Ultimate",
    "Adaptive_FixedCircuitBreaker",
    "Adaptive_NoTemporalMemory",
    "Adaptive_NoAdaptiveWeighting",
    "Adaptive_NoCircuitBreaker",
]



# Saída — bootstrap robusto do Google Drive
# Em Colab/Jupyter com Drive já montado, reutiliza /content/drive/MyDrive.
# Se a montagem for necessária, tenta montá-la e falha de forma explícita;
# não existe mais fallback silencioso para "./datasets".
if COLAB_DRIVE_MODE:
    DRIVE_ROOT = Path("/content/drive/MyDrive")

    if not DRIVE_ROOT.exists():
        if IN_COLAB:
            from google.colab import drive
            try:
                drive.mount("/content/drive", force_remount=False)
            except Exception as exc:
                raise RuntimeError(
                    "Falha ao montar o Google Drive em /content/drive. "
                    "O benchmark foi interrompido para evitar executar com caminhos locais incorretos."
                ) from exc
        else:
            raise RuntimeError(
                "COLAB_DRIVE_MODE=True, mas /content/drive/MyDrive não existe e "
                "google.colab não está disponível. Execute este notebook em um runtime "
                "com o Google Drive montado."
            )

    if not DRIVE_ROOT.exists():
        raise FileNotFoundError(
            "Google Drive não disponível após a tentativa de montagem: "
            f"{DRIVE_ROOT}"
        )

    PROJECT_ROOT = DRIVE_ROOT / PROJECT_ROOT_REL

    if not PROJECT_ROOT.exists():
        raise FileNotFoundError(
            "PROJECT_ROOT da campanha pós-banca não encontrado:\n"
            f"{PROJECT_ROOT}"
        )

    # Localiza o diretório real dos 9 CSVs dentro de 02_DADOS_HARMONIZADOS.
    # Primeiro testa o caminho oficial configurado; se ele mudou, procura de forma
    # determinística pelo primeiro arquivo esperado e valida a presença dos nove.
    configured_dataset_dir = DRIVE_ROOT / COLAB_DATASET_DIR_REL
    dataset_candidates = []

    if configured_dataset_dir.exists():
        dataset_candidates.append(configured_dataset_dir)

    harmonized_root = PROJECT_ROOT / "02_DADOS_HARMONIZADOS"
    if harmonized_root.exists():
        first_expected = EXPECTED_DEVICE_FILES[0]
        for match in harmonized_root.rglob(first_expected):
            candidate = match.parent
            if candidate not in dataset_candidates:
                dataset_candidates.append(candidate)

    valid_dataset_dirs = [
        candidate
        for candidate in dataset_candidates
        if all((candidate / fname).exists() for fname in EXPECTED_DEVICE_FILES)
    ]

    if not valid_dataset_dirs:
        checked = "\n".join(f" - {p}" for p in dataset_candidates) or " - nenhum candidato encontrado"
        raise FileNotFoundError(
            "Diretório contendo os 9 CSVs balanceados do N-BaIoT não encontrado.\n"
            "Candidatos verificados:\n"
            f"{checked}\n"
            f"Raiz pesquisada: {harmonized_root}"
        )

    # Preferência pelo caminho configurado; caso contrário, usa o primeiro candidato
    # completo encontrado sob 02_DADOS_HARMONIZADOS.
    DATASET_DIR = valid_dataset_dirs[0]
    OUTPUT_ROOT = DRIVE_ROOT / COLAB_RESULTS_REL

else:
    # Execução local só é permitida quando o usuário desligar explicitamente
    # COLAB_DRIVE_MODE e fornecer os dados na pasta ./datasets.
    DRIVE_ROOT = Path(".")
    PROJECT_ROOT = Path(".")
    DATASET_DIR = Path("datasets")
    OUTPUT_ROOT = Path("unified_results_csai4iot_pos_banca_2026")

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("\nBootstrap de caminhos validado:")
print(f"  DRIVE_ROOT   : {DRIVE_ROOT}")
print(f"  PROJECT_ROOT : {PROJECT_ROOT}")
print(f"  DATASET_DIR  : {DATASET_DIR}")
print(f"  OUTPUT_ROOT  : {OUTPUT_ROOT}")

# Persistir e exibir o perfil de pressão antes do benchmark
PRESSURE_PROFILE = validate_thesis_pressure_profile(N_MAIN_CLIENTS)
PRESSURE_PROFILE.to_csv(OUTPUT_ROOT / "thesis_pressure_profile.csv", index=False)
print("\nPerfil de pressão adversarial (S0..S16):")
print(PRESSURE_PROFILE.to_string(index=False))

CUMULATIVE_STRESS_PROFILE = validate_cumulative_stress_profiles()
CUMULATIVE_STRESS_PROFILE.to_csv(OUTPUT_ROOT / "cumulative_stress_profile.csv", index=False)
print("\nCumulative Stress Test — perfis de epsilon:")
print(
    CUMULATIVE_STRESS_PROFILE.groupby("Profile")["Epsilon"]
    .agg(["min", "max"])
    .to_string()
)

MIXED_DEVICE_SOURCE_MANIFEST = pd.DataFrame({
    "Order": list(range(1, len(EXPECTED_DEVICE_FILES) + 1)),
    "Source_File": EXPECTED_DEVICE_FILES,
    "Seed": VALIDATION_SEED,
    "Rounds": N_FEDERATED_ROUNDS,
    "Trust_Threshold": TRUST_THRESHOLD,
    "Clients": N_MAIN_CLIENTS,
    "Combined_Dataset": COMBINED_DATASET_NAME,
    "Source_As_Model_Feature": False,
})
MIXED_DEVICE_SOURCE_MANIFEST.to_csv(
    OUTPUT_ROOT / "mixed_device_source_manifest.csv",
    index=False
)


# ==============================================================================
# 2. UTILITÁRIOS
# ==============================================================================

def normalize_text(s: str) -> str:
    s = str(s)
    s = unicodedata.normalize("NFKD", s)
    s = "".join(c for c in s if not unicodedata.combining(c))
    s = s.lower()
    s = s.replace("-", "_").replace(" ", "_")
    s = re.sub(r"_+", "_", s)
    return s


def safe_slug(s: str, max_len: int = 120) -> str:
    s = str(s).strip()
    s = re.sub(r"\s+", "_", s)
    s = re.sub(r"[^A-Za-z0-9_\-]+", "", s)
    return s[:max_len] if s else "unknown"


def detect_dataset_name(filename: str, df: pd.DataFrame = None) -> str:
    text = filename
    if df is not None and "Dataset" in df.columns and df["Dataset"].notna().any():
        text += " " + str(df["Dataset"].dropna().iloc[0])

    s = normalize_text(text)
    original = str(text)

    if "dataset_a" in s or "cps_netflow" in s or "cps-netflow" in original.lower():
        return "A_CPS-NetFlow-IDS"
    if "dataset_b" in s or "packet_level" in s or "packet-level" in original.lower() or "botnet" in s:
        return "B_Packet-Level Botnet Set"
    if "dataset_c" in s or "mqtt_iot" in s or "mqtt-iot" in original.lower() or "mqtt" in s:
        return "C_MQTT-IoT-Flood"
    if "dataset_d" in s or "aggregated_traffic" in s or "aggregated traffic" in original.lower():
        return "D_Aggregated Traffic Set"

    return Path(filename).stem


def find_target_column(df: pd.DataFrame) -> str:
    candidates = ["class", "target", "label", "attack", "benign"]
    for c in df.columns:
        if c.lower() in candidates:
            return c
    return df.columns[-1]


def preprocess_dataframe(df: pd.DataFrame):
    cols_ignore = [
        "Dataset_Source",
        "dataset_source",
        "device_category",
        "host",
        "ts",
        "date",
        "time",
        "timestamp",
        "source_csv",
        "capture_id",
        "usage",
        "round_id",
        "Attack_Type",
        "attack_type"
    ]

    df = df.drop(columns=[c for c in cols_ignore if c in df.columns], errors="ignore")

    target_col = find_target_column(df)

    if df[target_col].dtype == object:
        df[target_col] = LabelEncoder().fit_transform(df[target_col].astype(str))

    y_series = df[target_col].astype(int)

    df_x = (
        df.drop(columns=[target_col])
        .select_dtypes(include=[np.number])
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0)
    )

    clean_df = pd.concat([df_x, y_series], axis=1)

    return clean_df, target_col



def canonical_feature_name(name: str) -> str:
    """Normaliza nomes para comparação entre fontes sem alterar o significado."""
    normalized = normalize_text(name)
    return FEATURE_ALIASES.get(normalized, normalized)


def harmonize_and_combine_datasets(datasets, seed=INITIAL_SEED):
    """
    Cria um cenário IoT multifonte usando apenas features numéricas semanticamente
    compatíveis por nome normalizado. O rótulo é padronizado para 'class'.

    Regras de segurança metodológica:
      1) não cria features artificiais para preencher esquemas incompatíveis;
      2) não usa test/eval para ajustar normalização (o scaler continua sendo fitado
         apenas dentro de run_single_seed, depois do split);
      3) opcionalmente balanceia o número de amostras por origem para evitar que
         uma fonte domine o cenário combinado;
      4) gera manifest para rastreabilidade.
    """
    if len(datasets) < 2:
        return None, pd.DataFrame([{
            "status": "skipped",
            "reason": "menos de dois datasets disponíveis"
        }])

    prepared = []
    feature_sets = []

    for ds in datasets:
        df = ds["df"].copy()
        target_col = ds["target_col"]

        feature_cols = [c for c in df.columns if c != target_col]
        rename_map = {c: canonical_feature_name(c) for c in feature_cols}

        # Evita colisões silenciosas após normalização/aliases.
        canonical_names = list(rename_map.values())
        if len(canonical_names) != len(set(canonical_names)):
            duplicates = sorted({x for x in canonical_names if canonical_names.count(x) > 1})
            raise ValueError(
                f"Colisão de features após normalização em {ds['dataset_name']}: {duplicates}"
            )

        tmp = df.rename(columns=rename_map).copy()
        tmp = tmp.rename(columns={target_col: "class"})

        feature_set = set(canonical_names)
        feature_sets.append(feature_set)
        prepared.append({
            "dataset_name": ds["dataset_name"],
            "filename": ds["filename"],
            "df": tmp,
            "features": feature_set,
        })

    common_features = sorted(set.intersection(*feature_sets))

    manifest_rows = []
    for item in prepared:
        manifest_rows.append({
            "Dataset_Source": item["dataset_name"],
            "Filename": item["filename"],
            "Rows_Original": len(item["df"]),
            "Features_Original": len(item["features"]),
            "Common_Features": len(common_features),
            "Common_Feature_List": "|".join(common_features),
        })

    manifest = pd.DataFrame(manifest_rows)

    if len(common_features) < MIN_COMMON_FEATURES:
        manifest["status"] = "blocked"
        manifest["reason"] = (
            f"interseção insuficiente: {len(common_features)} features comuns; "
            f"mínimo exigido={MIN_COMMON_FEATURES}"
        )
        return None, manifest

    if COMBINED_BALANCE_BY_SOURCE:
        n_per_source = min(len(item["df"]) for item in prepared)
    else:
        n_per_source = None

    parts = []
    for idx, item in enumerate(prepared):
        part = item["df"][common_features + ["class"]].copy()

        # Validação explícita para classificação binária do benchmark.
        classes = set(pd.Series(part["class"]).dropna().astype(int).unique().tolist())
        if not classes.issubset({0, 1}):
            raise ValueError(
                f"{item['dataset_name']} possui rótulos fora de 0/1 no cenário combinado: "
                f"{sorted(classes)}"
            )

        if n_per_source is not None and len(part) > n_per_source:
            # Amostragem estratificada aproximada por classe dentro de cada origem.
            sampled_parts = []
            counts = part["class"].value_counts(normalize=True)
            remaining = n_per_source

            for cls_idx, (cls, frac) in enumerate(counts.items()):
                if cls_idx == len(counts) - 1:
                    n_cls = remaining
                else:
                    n_cls = int(round(n_per_source * float(frac)))
                    n_cls = min(n_cls, int((part["class"] == cls).sum()))
                    remaining -= n_cls

                pool = part[part["class"] == cls]
                if n_cls > 0:
                    sampled_parts.append(
                        pool.sample(n=n_cls, random_state=seed + idx + int(cls))
                    )

            part = pd.concat(sampled_parts, ignore_index=True)
            if len(part) < n_per_source:
                missing = n_per_source - len(part)
                remaining_pool = item["df"][common_features + ["class"]].drop(
                    index=part.index, errors="ignore"
                )
                if len(remaining_pool) >= missing:
                    part = pd.concat([
                        part,
                        remaining_pool.sample(n=missing, random_state=seed + 1000 + idx)
                    ], ignore_index=True)

        parts.append(part)

    combined = pd.concat(parts, ignore_index=True)
    combined = combined.sample(frac=1.0, random_state=seed).reset_index(drop=True)

    manifest["Rows_Used"] = [len(p) for p in parts]
    manifest["status"] = "ok"
    manifest["reason"] = "harmonização por interseção de features comuns"

    return combined, manifest



def _iid_stratified_partition_indices(y, n_clients, seed):
    """Distribui cada classe aproximadamente de forma uniforme entre clientes."""
    rng = np.random.RandomState(seed)
    client_indices = [[] for _ in range(n_clients)]

    for cls in np.unique(y):
        idx = np.where(y == cls)[0].copy()
        rng.shuffle(idx)
        splits = np.array_split(idx, n_clients)
        for client_id, split in enumerate(splits):
            client_indices[client_id].extend(split.tolist())

    for client_id in range(n_clients):
        rng.shuffle(client_indices[client_id])

    return [np.asarray(idx, dtype=int) for idx in client_indices]


def _dirichlet_partition_indices(y, n_clients, seed, alpha, min_client_samples=MIN_CLIENT_SAMPLES):
    """Partição label-skew non-IID via Dirichlet.

    Para cada classe c, amostras são distribuídas aos clientes segundo
    p_c ~ Dirichlet(alpha). A rotina tenta garantir um número mínimo efetivo
    de amostras por cliente sem alterar os rótulos.
    """
    rng = np.random.RandomState(seed)
    n_total = len(y)
    effective_min = min(
        int(min_client_samples),
        max(2, n_total // max(1, n_clients * 4))
    )

    for _ in range(NONIID_MAX_ATTEMPTS):
        client_indices = [[] for _ in range(n_clients)]

        for cls in np.unique(y):
            idx = np.where(y == cls)[0].copy()
            rng.shuffle(idx)

            proportions = rng.dirichlet(np.full(n_clients, float(alpha)))
            counts = rng.multinomial(len(idx), proportions)

            start = 0
            for client_id, count in enumerate(counts):
                if count > 0:
                    client_indices[client_id].extend(
                        idx[start:start + count].tolist()
                    )
                start += count

        sizes = [len(v) for v in client_indices]
        if min(sizes) >= effective_min:
            for client_id in range(n_clients):
                rng.shuffle(client_indices[client_id])
            return [np.asarray(idx, dtype=int) for idx in client_indices]

    raise RuntimeError(
        "Não foi possível gerar partição Dirichlet respeitando o mínimo de "
        f"{effective_min} amostras/cliente após {NONIID_MAX_ATTEMPTS} tentativas. "
        "Ajuste NONIID_DIRICHLET_ALPHA ou MIN_CLIENT_SAMPLES."
    )


def partition_federated_clients(X, y, nodes, seed, partition_mode):
    """Cria shards uma única vez por seed e os reutiliza em todos os métodos."""
    n_clients = len(nodes)

    if partition_mode == "IID":
        indices = _iid_stratified_partition_indices(y, n_clients, seed)
        alpha_value = np.nan
    elif partition_mode == "NonIID_Dirichlet":
        indices = _dirichlet_partition_indices(
            y,
            n_clients,
            seed,
            alpha=NONIID_DIRICHLET_ALPHA
        )
        alpha_value = NONIID_DIRICHLET_ALPHA
    else:
        raise ValueError(f"Modo de particionamento desconhecido: {partition_mode}")

    shards_X = [X[idx].copy() for idx in indices]
    shards_y = [y[idx].copy() for idx in indices]

    class1_props = []
    rows = []
    for client_id, (node, y_client) in enumerate(zip(nodes, shards_y)):
        p1 = float(np.mean(y_client == MALICIOUS_CLASS)) if len(y_client) else np.nan
        class1_props.append(p1)
        rows.append({
            "Client_Index": client_id,
            "Client_ID": node,
            "Client_Samples": len(y_client),
            "Class1_Proportion": p1,
        })

    valid_props = np.asarray([v for v in class1_props if np.isfinite(v)], dtype=float)
    diagnostics = {
        "Partition_Mode": partition_mode,
        "Dirichlet_Alpha": alpha_value,
        "Client_Size_Min": int(min(len(v) for v in shards_y)),
        "Client_Size_Max": int(max(len(v) for v in shards_y)),
        "Client_Size_Mean": float(np.mean([len(v) for v in shards_y])),
        "Label1_Proportion_Std": float(np.std(valid_props)) if len(valid_props) else np.nan,
        "clients": rows,
    }

    return shards_X, shards_y, diagnostics


def get_attack_config(attack_name: str):
    if attack_name == "Cumulative":
        return {
            "canonical": "Cumulative",
            "benchmark": "Ataque_Cumulativo_SlowDrift",
            "nodes": list(DEFAULT_NODES),
            "stages": list(range(N_FEDERATED_ROUNDS)),
            "hidden_layers": HIDDEN_LAYERS_CUMULATIVE,
            "local_epochs": LOCAL_EPOCHS_CUMULATIVE
        }

    if attack_name == "Scaling":
        return {
            "canonical": "Scaling",
            "benchmark": "Ataque_ModelReplacement_Scaling",
            "nodes": list(DEFAULT_NODES),
            "stages": list(range(N_FEDERATED_ROUNDS)),
            "hidden_layers": HIDDEN_LAYERS_DEFAULT,
            "local_epochs": LOCAL_EPOCHS_DEFAULT
        }

    if attack_name == "Sybil":
        nodes = [f"Honest_{i}" for i in range(3)] + [f"Sybil_{i}" for i in range(7)]
        return {
            "canonical": "Sybil",
            "benchmark": "Ataque_Sybil_IdentityFlood",
            "nodes": nodes,
            "sybil_indices": list(range(3, 10)),
            "stages": list(range(N_FEDERATED_ROUNDS)),
            "hidden_layers": HIDDEN_LAYERS_DEFAULT,
            "local_epochs": LOCAL_EPOCHS_DEFAULT
        }

    if attack_name == "OnOff":
        return {
            "canonical": "OnOff",
            "benchmark": "Ataque_OnOff_SleepingCell",
            "nodes": list(DEFAULT_NODES),
            "sleeping_cell_indices": list(range(7)),
            "stages": list(range(N_FEDERATED_ROUNDS)),
            "hidden_layers": HIDDEN_LAYERS_DEFAULT,
            "local_epochs": LOCAL_EPOCHS_DEFAULT
        }

    raise ValueError(f"Ataque desconhecido: {attack_name}")


# ==============================================================================
# 3. MODELO E FL
# ==============================================================================

def create_model(seed, hidden_layers):
    return MLPClassifier(
        hidden_layer_sizes=hidden_layers,
        activation="relu",
        solver="adam",
        random_state=seed,
        max_iter=1
    )


def get_weights(model):
    return np.concatenate(
        [w.flatten() for w in model.coefs_] +
        [b.flatten() for b in model.intercepts_]
    )


def set_weights(model, flat_weights, ref_model):
    idx = 0

    for i in range(len(model.coefs_)):
        shape = ref_model.coefs_[i].shape
        size = np.prod(shape)
        model.coefs_[i] = flat_weights[idx:idx + size].reshape(shape)
        idx += size

    for i in range(len(model.intercepts_)):
        shape = ref_model.intercepts_[i].shape
        size = np.prod(shape)
        model.intercepts_[i] = flat_weights[idx:idx + size].reshape(shape)
        idx += size


def local_training(global_weights, X, y, classes, ref_model, seed, hidden_layers, local_epochs):
    model = create_model(seed, hidden_layers)

    if len(X) >= 5:
        model.partial_fit(X[:5], y[:5], classes=classes)
    else:
        model.partial_fit(X, y, classes=classes)

    set_weights(model, global_weights.copy(), ref_model)

    rng = np.random.RandomState(seed)

    for _ in range(local_epochs):
        p = rng.permutation(len(X))
        model.partial_fit(X[p], y[p], classes=classes)

    return get_weights(model) - global_weights


# ==============================================================================
# 4. ATAQUE E AGREGAÇÃO
# ==============================================================================

def get_attack_state(attack_name, stage_idx, node_idx, config):
    """
    Projeta os 100 rounds na escala legada S0..S16 e aplica o perfil de
    pressão adversarial usado na tese.

    - Cumulative: 1 cliente persistente; a pressão cresce pela duração acumulada
      da deriva lenta (epsilon fixo), não pela cardinalidade do atacante.
    - Scaling: 10% -> 30% -> 50% -> 70% de clientes comprometidos, com
      model-replacement usando fator de escala fixo gamma=100.
    - Sybil: número de identidades ativas cresce segundo o mesmo perfil,
      culminando em 7 Sybils vs 3 honestos em S12..S16.
    - OnOff: população comprometida cresce segundo o perfil, mas só injeta
      poisoning nas janelas ON definidas em THESIS_ONOFF_ACTIVE_STAGES.
    """
    nodes = config["nodes"]
    n_nodes = len(nodes)
    legacy_stage = round_to_legacy_stage(stage_idx)
    pressure_fraction = pressure_fraction_for_legacy_stage(legacy_stage)
    pressure_attackers = pressure_attackers_for_legacy_stage(legacy_stage, n_nodes)

    attack_active = False
    is_attacker = False
    active_attackers = 0
    compromised_clients = 0
    attack_note = "inactive"

    if attack_name == "Cumulative":
        attack_active = legacy_stage >= CUMULATIVE_ATTACK_START_STAGE
        is_attacker = (node_idx == 0) and attack_active
        compromised_clients = 1 if attack_active else 0
        active_attackers = compromised_clients
        attack_note = "cumulative_slow_drift" if attack_active else "warmup"

    elif attack_name == "Scaling":
        compromised_clients = pressure_attackers
        active_attackers = compromised_clients
        attack_active = active_attackers > 0
        is_attacker = node_idx < active_attackers
        attack_note = f"scaling_rho_{pressure_fraction:.2f}_attackers_{active_attackers}"

    elif attack_name == "Sybil":
        sybil_indices = list(config["sybil_indices"])
        compromised_clients = min(pressure_attackers, len(sybil_indices))
        active_sybil_indices = set(sybil_indices[:compromised_clients])
        active_attackers = compromised_clients
        attack_active = active_attackers > 0
        is_attacker = node_idx in active_sybil_indices
        attack_note = f"sybil_rho_{pressure_fraction:.2f}_identities_{active_attackers}"

    elif attack_name == "OnOff":
        sleeping_indices = list(config["sleeping_cell_indices"])
        compromised_clients = min(pressure_attackers, len(sleeping_indices))
        active_pool = set(sleeping_indices[:compromised_clients])
        attack_active = (legacy_stage in THESIS_ONOFF_ACTIVE_STAGES) and compromised_clients > 0
        active_attackers = compromised_clients if attack_active else 0
        is_attacker = (node_idx in active_pool) and attack_active
        attack_note = (
            f"onoff_ON_rho_{pressure_fraction:.2f}_attackers_{active_attackers}"
            if attack_active else
            f"onoff_SLEEP_rho_{pressure_fraction:.2f}_compromised_{compromised_clients}"
        )

    return {
        "attack_active": attack_active,
        "is_attacker": is_attacker,
        "active_attackers": active_attackers,
        "compromised_clients": compromised_clients,
        "pressure_fraction": pressure_fraction,
        "attack_note": attack_note,
        "legacy_stage": legacy_stage
    }

def select_trigger_from_training(X_train, y_train):
    """Seleciona uma feature pouco discriminativa e um valor de trigger in-distribution.

    A escolha usa SOMENTE o conjunto de treino para evitar leakage do conjunto de avaliação.
    Critério da feature: menor diferença padronizada entre as médias das classes 0 e 1,
    entre features não constantes e com cardinalidade suficiente.
    Valor do trigger: quantil alto (TRIGGER_QUANTILE) da própria feature no treino.
    """
    X_train = np.asarray(X_train, dtype=float)
    y_train = np.asarray(y_train)

    benign = X_train[y_train == BENIGN_CLASS]
    malicious = X_train[y_train == MALICIOUS_CLASS]
    if len(benign) == 0 or len(malicious) == 0:
        raise ValueError("Treino precisa conter classes benignas e maliciosas para selecionar trigger.")

    candidates = []
    for j in range(X_train.shape[1]):
        col = X_train[:, j]
        finite = col[np.isfinite(col)]
        if len(finite) == 0:
            continue
        if np.nanstd(finite) <= 1e-12:
            continue
        if len(np.unique(finite)) < TRIGGER_MIN_UNIQUE_VALUES:
            continue

        b = benign[:, j]
        m = malicious[:, j]
        pooled_std = np.nanstd(col) + 1e-12
        separation = abs(np.nanmean(b) - np.nanmean(m)) / pooled_std
        trigger_value = float(np.nanquantile(finite, TRIGGER_QUANTILE))
        candidates.append((separation, j, trigger_value))

    if not candidates:
        raise ValueError("Nenhuma feature elegível para construção do trigger.")

    candidates.sort(key=lambda x: (x[0], x[1]))
    separation, feature_idx, trigger_value = candidates[0]
    return {
        "feature_idx": int(feature_idx),
        "value": float(trigger_value),
        "separation": float(separation),
        "quantile": float(TRIGGER_QUANTILE),
    }


def apply_trigger_raw(X, trigger_spec):
    """Aplica o trigger no espaço bruto sem alterar rótulos."""
    X_t = np.asarray(X).copy()
    if len(X_t) and X_t.shape[1] > 0:
        X_t[:, trigger_spec["feature_idx"]] = trigger_spec["value"]
    return X_t


def apply_local_poisoning(
    X_n,
    y_n,
    seed,
    stage_idx,
    node_idx,
    trigger_spec,
    source_class=BACKDOOR_SOURCE_CLASS,
    target_class=BACKDOOR_TARGET_CLASS,
):
    """Aplica poisoning direcionado de evasão.

    Somente amostras maliciosas (classe-fonte 1) são candidatas.
    Uma fração POISON_RATE recebe o trigger e tem o rótulo alterado para benigno (0).
    """
    source_idx = np.flatnonzero(y_n == source_class)
    if len(source_idx) == 0:
        return X_n, y_n

    n_p = int(len(source_idx) * POISON_RATE)
    if POISON_RATE > 0 and n_p == 0:
        n_p = 1
    n_p = min(n_p, len(source_idx))

    if n_p > 0:
        rng = np.random.RandomState(seed + stage_idx + node_idx)
        p_idx = rng.choice(source_idx, n_p, replace=False)
        if X_n.shape[1] > 0:
            X_n[p_idx, trigger_spec["feature_idx"]] = trigger_spec["value"]
        y_n[p_idx] = target_class

    return X_n, y_n


def validate_asr_semantics():
    """Fail-fast para impedir inversão acidental da direção do ASR."""
    assert BENIGN_CLASS == 0, "Esperado BENIGN_CLASS=0."
    assert MALICIOUS_CLASS == 1, "Esperado MALICIOUS_CLASS=1."
    assert BACKDOOR_SOURCE_CLASS == MALICIOUS_CLASS
    assert BACKDOOR_TARGET_CLASS == BENIGN_CLASS
    assert BACKDOOR_SOURCE_CLASS != BACKDOOR_TARGET_CLASS
    return True


def train_clean_control_model(
    X_train, y_train, shards_X, shards_y, scaler, seed,
    hidden_layers, local_epochs, classes, rounds
):
    """Treina um controle limpo FedAvg, sem poisoning e sem clientes maliciosos."""
    model = create_model(seed, hidden_layers)
    n_init = min(10, len(X_train))
    model.partial_fit(
        scaler.transform(X_train[:n_init]),
        y_train[:n_init],
        classes=classes
    )
    weights = get_weights(model)
    sample_counts = np.array([len(y) for y in shards_y], dtype=float)
    total = float(sample_counts.sum()) if sample_counts.sum() > 0 else 1.0

    for _ in range(rounds):
        updates = []
        for X_n, y_n in zip(shards_X, shards_y):
            delta = local_training(
                weights,
                scaler.transform(X_n),
                y_n,
                classes,
                model,
                seed,
                hidden_layers,
                local_epochs,
            )
            updates.append(delta)
        agg = np.zeros_like(weights)
        for delta, n in zip(updates, sample_counts):
            agg += delta * (n / total)
        weights += agg
        set_weights(model, weights, model)
    return model


# Validação semântica executada na carga do benchmark.
validate_asr_semantics()


def _adaptive_similarity(local_update, root_delta, root_norm):
    """Calcula a evidência instantânea usada pela defesa adaptativa."""
    norm_factor = root_norm / (np.linalg.norm(local_update) + 1e-12)
    d_norm = local_update * norm_factor
    sim = np.dot(root_delta, d_norm) / (
        root_norm * (np.linalg.norm(d_norm) + 1e-12)
    )
    return float(sim)


def _update_temporal_trust(sim, old_trust, defense_config):
    """Atualiza o estado temporal de confiança.

    Full: reproduz a regra original com penalização rápida e recuperação lenta.
    Sem memória: usa somente a evidência instantânea do round atual.
    """
    raw_score = max(0.0, sim)

    if not defense_config.get("use_temporal_memory", True):
        new_trust = 0.0 if sim < 0 else raw_score
        return raw_score, float(np.clip(new_trust, 0.0, 1.0))

    if sim < 0:
        new_trust = 0.0
    elif (raw_score - old_trust) < 0:
        new_trust = old_trust + (ALPHA_PENALTY * (raw_score - old_trust))
    else:
        new_trust = old_trust + (BETA_RECOVERY * (raw_score - old_trust))

    return raw_score, float(np.clip(new_trust, 0.0, 1.0))


def _adaptive_client_weight(trust_value, defense_config):
    """Converte confiança em participação/peso de agregação."""
    if trust_value < TRUST_THRESHOLD:
        return 0.0, False

    if defense_config.get("use_adaptive_weighting", True):
        return float(trust_value), True

    # Ablação da ponderação: clientes aceitos recebem peso uniforme.
    return 1.0, True


def _init_circuit_breaker_state():
    """Estado temporal do Circuit Breaker adaptativo por estratégia/execução."""
    return {
        "calibration_mean_trust": [],
        "baseline_mean_trust": np.nan,
        "critical_streak": 0,
        "healthy_streak": 0,
        "latched": False,
        "last_mode": "INIT",
        "last_lambda": 1.0,
    }


def _effective_client_count(scores):
    """Effective Sample Size dos scores positivos de confiança."""
    w = np.asarray([float(v) for v in scores if float(v) > 0], dtype=float)
    if len(w) == 0:
        return 0.0
    return float((w.sum() ** 2) / (np.square(w).sum() + 1e-12))


def _candidate_trusted_aggregate(local_updates, trust_w, root_norm, g_weights):
    """Agrega apenas clientes aceitos, antes da decisão do Circuit Breaker."""
    candidate = np.zeros_like(g_weights)
    total = 0.0
    for i, w in enumerate(trust_w):
        if w > 0:
            norm_up = local_updates[i] * (
                root_norm / (np.linalg.norm(local_updates[i]) + 1e-12)
            )
            candidate += float(w) * norm_up
            total += float(w)
    if total > 0:
        candidate /= total
    return candidate, float(total)


def _adaptive_circuit_breaker_decision(
    valid_cnt,
    trusted_scores,
    candidate_update,
    root_delta,
    root_norm,
    n_nodes,
    round_idx,
    cb_state,
):
    """Decisão NORMAL / DEGRADED / BREAK sem conhecer o tipo de ataque.

    NORMAL: existe maioria confiável.
    DEGRADED: abaixo da maioria, mas ainda há suporte mínimo, diversidade efetiva
    e alinhamento não adversarial com a referência. A atualização é atenuada.
    BREAK: suporte insuficiente de forma persistente; a atualização é congelada.
    """
    if cb_state is None:
        cb_state = _init_circuit_breaker_state()

    majority = (int(n_nodes) // 2) + 1
    min_trusted = max(2, int(np.ceil(CB_MIN_TRUSTED_FRACTION * n_nodes)))
    min_ess = max(2, int(np.ceil(CB_MIN_EFFECTIVE_FRACTION * n_nodes)))

    positive_scores = np.asarray([float(v) for v in trusted_scores if float(v) > 0], dtype=float)
    trust_mass = float(positive_scores.sum() / max(1, n_nodes))
    mean_trust = float(positive_scores.mean()) if len(positive_scores) else 0.0
    ess = _effective_client_count(positive_scores)

    cand_norm = np.linalg.norm(candidate_update)
    if cand_norm > 1e-12 and root_norm > 1e-12:
        alignment = float(np.dot(candidate_update, root_delta) / (cand_norm * root_norm + 1e-12))
    else:
        alignment = 0.0

    calibrating = int(round_idx) < CB_CALIBRATION_ROUNDS
    if calibrating:
        if mean_trust > 0:
            cb_state["calibration_mean_trust"].append(mean_trust)
            cb_state["baseline_mean_trust"] = float(np.median(cb_state["calibration_mean_trust"]))
        cb_state["critical_streak"] = 0
        cb_state["healthy_streak"] = 0
        cb_state["latched"] = bool(valid_cnt == 0)
        mode = "BREAK" if cb_state["latched"] else "CALIBRATION"
        lam = 0.0 if cb_state["latched"] else 1.0
        reason = "adaptive_calibration_no_trusted" if cb_state["latched"] else "adaptive_calibration"
    else:
        baseline_mean = float(cb_state.get("baseline_mean_trust", np.nan))
        if not np.isfinite(baseline_mean) or baseline_mean <= 1e-12:
            baseline_mean = max(mean_trust, TRUST_THRESHOLD)
            cb_state["baseline_mean_trust"] = baseline_mean

        normal = valid_cnt >= majority
        degraded_support = (
            valid_cnt >= min_trusted
            and ess >= min_ess
            and alignment >= CB_ALIGNMENT_THRESHOLD
        )
        critical = not degraded_support

        if critical:
            cb_state["critical_streak"] = int(cb_state.get("critical_streak", 0)) + 1
            cb_state["healthy_streak"] = 0
        else:
            cb_state["healthy_streak"] = int(cb_state.get("healthy_streak", 0)) + 1
            cb_state["critical_streak"] = 0

        if cb_state.get("latched", False):
            if (not critical) and cb_state["healthy_streak"] >= CB_RELEASE_PATIENCE:
                cb_state["latched"] = False
        else:
            if valid_cnt == 0 or cb_state["critical_streak"] >= CB_TRIGGER_PATIENCE:
                cb_state["latched"] = True

        if cb_state["latched"]:
            mode = "BREAK"
            lam = 0.0
            reason = "adaptive_break_insufficient_support"
        elif normal:
            mode = "NORMAL"
            lam = 1.0
            reason = "adaptive_normal_majority"
        else:
            mode = "DEGRADED"
            count_ratio = float(valid_cnt / max(1, majority))
            trust_ratio = float(np.clip(mean_trust / (baseline_mean + 1e-12), 0.0, 1.0))
            lam = float(np.clip(count_ratio * trust_ratio, CB_MIN_LAMBDA, 1.0))
            reason = "adaptive_degraded_trusted_minority"

    cb_state["last_mode"] = mode
    cb_state["last_lambda"] = float(lam)

    return {
        "mode": mode,
        "lambda": float(lam),
        "reason": reason,
        "majority": int(majority),
        "min_trusted": int(min_trusted),
        "min_ess": int(min_ess),
        "trust_mass": float(trust_mass),
        "mean_trust": float(mean_trust),
        "ess": float(ess),
        "alignment": float(alignment),
        "calibrating": int(calibrating),
        "critical_streak": int(cb_state.get("critical_streak", 0)),
        "healthy_streak": int(cb_state.get("healthy_streak", 0)),
        "baseline_mean_trust": float(cb_state.get("baseline_mean_trust", np.nan)),
    }


def aggregate_updates(
    strategy,
    local_updates,
    root_delta,
    root_norm,
    g_weights,
    trust_memory,
    nodes,
    defense_config=None,
    client_sample_counts=None,
    round_idx=0,
    cb_state=None,
):
    agg_update = np.zeros_like(g_weights)
    defense_config = defense_config or DEFENSE_CONFIG

    if strategy == "FedAvg":
        # FedAvg canônico: média ponderada pelo número de amostras locais.
        # Isto é essencial sob partições non-IID/Dirichlet, nas quais os shards
        # podem ter tamanhos substancialmente diferentes.
        if client_sample_counts is None:
            weights = np.ones(len(local_updates), dtype=float)
        else:
            weights = np.asarray(client_sample_counts, dtype=float)
            if len(weights) != len(local_updates) or np.sum(weights) <= 0:
                raise ValueError("client_sample_counts inválido para FedAvg.")
        return (
            np.average(np.stack(local_updates), axis=0, weights=weights),
            trust_memory, 0, len(local_updates), []
        )

    if strategy == "Krum":
        k = max(1, len(local_updates) - 3)
        dists = cdist(np.stack(local_updates), np.stack(local_updates))
        scores = [np.sum(np.sort(d)[1:k + 1]) for d in dists]
        return local_updates[np.argmin(scores)], trust_memory, 0, 1, []

    if strategy == "FLAME":
        stk = np.stack(local_updates)
        dists = cosine_distances(stk)

        labels = hdbscan.HDBSCAN(
            metric="precomputed",
            min_cluster_size=2,
            allow_single_cluster=True
        ).fit_predict(dists)

        u, c = np.unique(labels, return_counts=True)
        valid = u[u != -1]

        if len(valid) > 0:
            valid_counts = {label: np.sum(labels == label) for label in valid}
            best_label = max(valid_counts, key=valid_counts.get)
            sel = stk[labels == best_label]
        else:
            sel = stk

        norms = np.linalg.norm(sel, axis=1)
        med = np.median(norms) if len(norms) else 0.0

        clip = [
            v * min(1, med / (np.linalg.norm(v) + 1e-12))
            for v in sel
        ]

        agg_update = (
            np.mean(clip, axis=0) +
            np.random.normal(0, FLAME_NOISE_SIGMA * med, g_weights.shape)
        )

        return agg_update, trust_memory, 0, len(sel), []

    if strategy == "FLTrust":
        scores = []

        for d in local_updates:
            sim = np.dot(root_delta, d) / (
                root_norm * (np.linalg.norm(d) + 1e-12)
            )

            score = max(0, sim)
            scores.append(score)

            norm_up = d * (
                root_norm / (np.linalg.norm(d) + 1e-12)
            )

            agg_update += score * norm_up

        if sum(scores) > 0:
            agg_update /= sum(scores)

        trusted_clients = int(np.sum(np.array(scores) > 0))
        return agg_update, trust_memory, 0, trusted_clients, []

    if strategy == "Adaptive_Ultimate":
        valid_cnt = 0
        maj_thr = (len(nodes) // 2) + 1
        trust_w = []
        client_diagnostics = []

        for i, d in enumerate(local_updates):
            sim = _adaptive_similarity(d, root_delta, root_norm)
            old = float(trust_memory[nodes[i]])
            raw_score, new = _update_temporal_trust(sim, old, defense_config)

            # Mantemos o estado atualizado mesmo na variante sem memória; nesse caso
            # ele representa apenas o valor instantâneo e não influencia o round seguinte.
            trust_memory[nodes[i]] = new

            w, is_trusted = _adaptive_client_weight(new, defense_config)
            if is_trusted:
                valid_cnt += 1
            trust_w.append(w)

            client_diagnostics.append({
                "Client": nodes[i],
                "Similarity": sim,
                "Raw_Score": raw_score,
                "Trust_Before": old,
                "Trust_After": new,
                "Adaptive_Weight": w,
                "Client_Trusted": int(is_trusted),
            })

        circuit_breaker = 0
        cb_enabled = defense_config.get("use_circuit_breaker", True)
        cb_mode = defense_config.get("circuit_breaker_mode", "fixed")

        # Primeiro calculamos o agregado candidato entre os clientes aceitos.
        # A decisão do CB atua depois sobre esse candidato: permitir, atenuar ou congelar.
        candidate_update, candidate_weight_sum = _candidate_trusted_aggregate(
            local_updates, trust_w, root_norm, g_weights
        )
        trusted_scores = [
            float(row["Trust_After"]) if int(row["Client_Trusted"]) == 1 else 0.0
            for row in client_diagnostics
        ]

        cb_diag = {
            "mode": "DISABLED",
            "lambda": 1.0,
            "reason": "disabled",
            "majority": maj_thr,
            "min_trusted": 0,
            "min_ess": 0,
            "trust_mass": float(np.sum(trusted_scores) / max(1, len(nodes))),
            "mean_trust": float(np.mean([v for v in trusted_scores if v > 0])) if any(v > 0 for v in trusted_scores) else 0.0,
            "ess": _effective_client_count(trusted_scores),
            "alignment": 0.0,
            "calibrating": 0,
            "critical_streak": 0,
            "healthy_streak": 0,
            "baseline_mean_trust": np.nan,
        }

        if not cb_enabled or cb_mode == "disabled":
            agg_update = candidate_update
        elif cb_mode == "adaptive":
            cb_diag = _adaptive_circuit_breaker_decision(
                valid_cnt=valid_cnt,
                trusted_scores=trusted_scores,
                candidate_update=candidate_update,
                root_delta=root_delta,
                root_norm=root_norm,
                n_nodes=len(nodes),
                round_idx=round_idx,
                cb_state=cb_state,
            )
            circuit_breaker = int(cb_diag["mode"] == "BREAK")
            agg_update = candidate_update * cb_diag["lambda"]
        else:
            # Baseline original da tese: maioria fixa após filtragem por confiança.
            if valid_cnt < maj_thr:
                circuit_breaker = 1
                agg_update = np.zeros_like(g_weights)
                cb_diag.update({
                    "mode": "FIXED_BREAK",
                    "lambda": 0.0,
                    "reason": "trusted_clients_below_majority",
                })
            else:
                agg_update = candidate_update
                cb_diag.update({
                    "mode": "FIXED_NORMAL",
                    "lambda": 1.0,
                    "reason": "not_triggered",
                })

        cb_reason = cb_diag["reason"]
        for row in client_diagnostics:
            row["Trusted_Clients"] = valid_cnt
            row["Majority_Threshold"] = maj_thr
            row["Circuit_Breaker_Enabled"] = int(cb_enabled)
            row["Circuit_Breaker_Mode"] = cb_mode
            row["Circuit_Breaker"] = circuit_breaker
            row["Circuit_Breaker_Reason"] = cb_reason
            row["CB_State"] = cb_diag["mode"]
            row["CB_Lambda"] = cb_diag["lambda"]
            row["CB_Trust_Mass"] = cb_diag["trust_mass"]
            row["CB_Mean_Trusted_Trust"] = cb_diag["mean_trust"]
            row["CB_Effective_Clients"] = cb_diag["ess"]
            row["CB_Aggregate_Alignment"] = cb_diag["alignment"]
            row["CB_Min_Trusted"] = cb_diag["min_trusted"]
            row["CB_Min_ESS"] = cb_diag["min_ess"]
            row["CB_Calibrating"] = cb_diag["calibrating"]
            row["CB_Critical_Streak"] = cb_diag["critical_streak"]
            row["CB_Healthy_Streak"] = cb_diag["healthy_streak"]
            row["CB_Baseline_Mean_Trust"] = cb_diag["baseline_mean_trust"]

        return agg_update, trust_memory, circuit_breaker, valid_cnt, client_diagnostics

    raise ValueError(f"Estratégia desconhecida: {strategy}")

# ==============================================================================
# 5. EXECUÇÃO DE UM ATAQUE (stage_idx = round federado, mantido por compatibilidade)
# ==============================================================================
# 5. EXECUÇÃO DE UM ATAQUE
# ==============================================================================

def run_single_seed(df, target_col, seed, dataset_name, attack_name, partition_mode, cumulative_profile=None):
    config = get_attack_config(attack_name)
    if attack_name == "Cumulative":
        if cumulative_profile is None:
            cumulative_profile = "Base"
        if cumulative_profile not in CUMULATIVE_STRESS_PROFILES:
            raise ValueError(f"Perfil Cumulative inválido: {cumulative_profile}")
        attack_label = f"Cumulative_{cumulative_profile}"
        benchmark_label = f"{config['benchmark']}_{cumulative_profile}"
    else:
        attack_label = attack_name
        benchmark_label = config["benchmark"]

    nodes = config["nodes"]
    stages = config["stages"]
    hidden_layers = config["hidden_layers"]
    local_epochs = config["local_epochs"]

    y_raw = df[target_col].values.astype(int)
    X_raw = df.drop(columns=[target_col]).values
    classes = np.unique(y_raw)

    X_train, X_test, y_train, y_test = train_test_split(
        X_raw,
        y_raw,
        test_size=0.2,
        stratify=y_raw,
        random_state=seed
    )

    X_root, X_eval, y_root, y_eval = train_test_split(
        X_test,
        y_test,
        test_size=0.5,
        stratify=y_test,
        random_state=seed
    )

    scaler = StandardScaler().fit(X_train)

    X_root_sc = scaler.transform(X_root)
    X_eval_sc = scaler.transform(X_eval)

    # A partição é gerada uma única vez por seed/modo e reutilizada em TODAS
    # as estratégias e rounds, assegurando comparação pareada.
    shards_X, shards_y, partition_diag = partition_federated_clients(
        X_train,
        y_train,
        nodes,
        seed,
        partition_mode
    )

    client_sample_counts = [len(y_part) for y_part in shards_y]

    # Trigger definido apenas a partir do treino.
    trigger_spec = select_trigger_from_training(X_train, y_train)

    # Backdoor test set para evasão: amostras MALICIOSAS + trigger.
    #
    # Semântica do ASR:
    #   source = MALICIOUS_CLASS (1)
    #   target = BENIGN_CLASS (0)
    #
    # O trigger é aplicado no MESMO domínio usado no poisoning:
    #   1) selecionar amostras maliciosas no espaço bruto;
    #   2) aplicar o trigger bruto;
    #   3) transformar com o scaler ajustado apenas no treino;
    #   4) medir a fração classificada como BENIGN_CLASS.
    mask_backdoor_source = (y_eval == BACKDOOR_SOURCE_CLASS)

    if np.sum(mask_backdoor_source) > 0:
        X_source_raw = X_eval[mask_backdoor_source].copy()
        X_source_sc = scaler.transform(X_source_raw)
        X_backdoor_raw = apply_trigger_raw(X_source_raw, trigger_spec)
        X_backdoor = scaler.transform(X_backdoor_raw)
    else:
        X_source_raw = np.empty((0, X_eval.shape[1]))
        X_source_sc = np.empty((0, X_eval_sc.shape[1]))
        X_backdoor_raw = np.empty((0, X_eval.shape[1]))
        X_backdoor = np.empty((0, X_eval_sc.shape[1]))

    # Controle limpo: mede o efeito do trigger sem poisoning.
    clean_control_model = train_clean_control_model(
        X_train=X_train,
        y_train=y_train,
        shards_X=shards_X,
        shards_y=shards_y,
        scaler=scaler,
        seed=seed,
        hidden_layers=hidden_layers,
        local_epochs=local_epochs,
        classes=classes,
        rounds=N_FEDERATED_ROUNDS,
    )
    clean_source_error = (
        float(np.mean(clean_control_model.predict(X_source_sc) == BACKDOOR_TARGET_CLASS))
        if len(X_source_sc) > 0 else 0.0
    )
    clean_trigger_asr = (
        float(np.mean(clean_control_model.predict(X_backdoor) == BACKDOOR_TARGET_CLASS))
        if len(X_backdoor) > 0 else 0.0
    )
    clean_trigger_delta = clean_trigger_asr - clean_source_error

    print(
        f"      Trigger control | feature={trigger_spec['feature_idx']} "
        f"q={trigger_spec['quantile']:.2f} value={trigger_spec['value']:.6g} | "
        f"NaturalFN={clean_source_error:.4f} | "
        f"ASR_clean_trigger={clean_trigger_asr:.4f} | "
        f"Delta={clean_trigger_delta:.4f}"
    )

    if clean_trigger_asr > TRIGGER_MAX_CLEAN_ASR or clean_trigger_delta > TRIGGER_MAX_CLEAN_DELTA:
        raise AssertionError(
            "Trigger inválido: o trigger sozinho já induz evasão excessiva no controle limpo. "
            f"ASR_clean_trigger={clean_trigger_asr:.4f} (limite={TRIGGER_MAX_CLEAN_ASR:.4f}), "
            f"Delta={clean_trigger_delta:.4f} (limite={TRIGGER_MAX_CLEAN_DELTA:.4f}). "
            "Não usar este run como resultado científico."
        )

    results = []
    client_logs = []

    print(f"\n   Seed {seed} | Attack={attack_label} | Dataset={dataset_name} | Partition={partition_mode}")

    for strategy in STRATEGIES:
        strategy_start_time = time.perf_counter()

        if strategy in ADAPTIVE_VARIANT_TO_CONFIG:
            ablation_variant, strategy_defense_config = ADAPTIVE_VARIANT_TO_CONFIG[strategy]
            base_strategy = "Adaptive_Ultimate"
        else:
            ablation_variant = "ExternalBaseline"
            strategy_defense_config = DEFENSE_CONFIG
            base_strategy = strategy

        g_model = create_model(seed, hidden_layers)
        # Inicialização exclusivamente com dados de treino para evitar leakage
        # do conjunto de avaliação no estado inicial do modelo global.
        n_init = min(10, len(X_train))
        X_init_sc = scaler.transform(X_train[:n_init])
        y_init = y_train[:n_init]
        g_model.partial_fit(X_init_sc, y_init, classes=classes)

        g_weights = get_weights(g_model)
        trust_memory = {n: 0.5 for n in nodes}
        cb_state = _init_circuit_breaker_state()

        # Rastreamento específico do ASR por estratégia para separar
        # períodos de ataque ativo/inativo e identificar o pico.
        strategy_asr_values = []
        strategy_asr_active_values = []
        strategy_asr_inactive_values = []

        for stage_idx in stages:
            stage_start_time = time.perf_counter()
            stage_name = f"S{stage_idx}"

            local_updates = []
            client_attack_states = []
            active_attackers_stage = 0
            attack_active_stage = False
            attack_note = "inactive"
            cumulative_epsilon_stage = (
                cumulative_epsilon_for_round(stage_idx, cumulative_profile)
                if attack_name == "Cumulative" else np.nan
            )

            for i, node in enumerate(nodes):
                X_n = shards_X[i].copy()
                y_n = shards_y[i].copy()

                attack_state = get_attack_state(attack_name, stage_idx, i, config)
                client_attack_states.append(attack_state.copy())

                if attack_state["attack_active"]:
                    attack_active_stage = True
                    active_attackers_stage = max(active_attackers_stage, attack_state["active_attackers"])
                    attack_note = attack_state["attack_note"]

                if attack_state["is_attacker"] and attack_name == "Cumulative":
                    # Eq. (2.7) da tese:
                    # delta_mal^(t) = delta_honest^(t) + epsilon * grad_drift_t
                    #
                    # grad_drift_t é obtido de forma reproduzível pela diferença
                    # entre a atualização local envenenada e a atualização limpa
                    # do MESMO cliente, na MESMA rodada. Isso mantém a perturbação
                    # próxima da trajetória legítima e evita o erro anterior de
                    # simplesmente multiplicar todo o update por epsilon.
                    X_clean = X_n.copy()
                    y_clean = y_n.copy()

                    delta_honest = local_training(
                        g_weights,
                        scaler.transform(X_clean),
                        y_clean,
                        classes,
                        g_model,
                        seed,
                        hidden_layers,
                        local_epochs
                    )

                    X_poison = X_clean.copy()
                    y_poison = y_clean.copy()
                    X_poison, y_poison = apply_local_poisoning(
                        X_poison,
                        y_poison,
                        seed,
                        stage_idx,
                        i,
                        trigger_spec
                    )

                    delta_poison = local_training(
                        g_weights,
                        scaler.transform(X_poison),
                        y_poison,
                        classes,
                        g_model,
                        seed,
                        hidden_layers,
                        local_epochs
                    )

                    drift_direction = delta_poison - delta_honest
                    delta = delta_honest + (cumulative_epsilon_stage * drift_direction)

                else:
                    if attack_state["is_attacker"]:
                        X_n, y_n = apply_local_poisoning(
                            X_n,
                            y_n,
                            seed,
                            stage_idx,
                            i,
                            trigger_spec
                        )

                    delta = local_training(
                        g_weights,
                        scaler.transform(X_n),
                        y_n,
                        classes,
                        g_model,
                        seed,
                        hidden_layers,
                        local_epochs
                    )

                    # Pós-processamento específico do Model Replacement.
                    if attack_state["is_attacker"] and attack_name == "Scaling":
                        delta = delta * SCALING_FACTOR

                local_updates.append(delta)

            root_delta = local_training(
                g_weights,
                X_root_sc,
                y_root,
                classes,
                g_model,
                seed,
                hidden_layers,
                local_epochs
            )

            root_norm = np.linalg.norm(root_delta) + 1e-12

            agg_update, trust_memory, circuit_breaker, trusted_clients, defense_diagnostics = aggregate_updates(
                strategy=base_strategy,
                local_updates=local_updates,
                root_delta=root_delta,
                root_norm=root_norm,
                g_weights=g_weights,
                trust_memory=trust_memory,
                nodes=nodes,
                defense_config=strategy_defense_config,
                client_sample_counts=client_sample_counts,
                round_idx=stage_idx,
                cb_state=cb_state,
            )

            for diag_idx, drow in enumerate(defense_diagnostics):
                client_attack_state = client_attack_states[diag_idx]
                client_logs.append({
                    "Benchmark": benchmark_label,
                    "Attack": attack_label,
                    "Cumulative_Profile": cumulative_profile if attack_name == "Cumulative" else "",
                    "Cumulative_Epsilon": cumulative_epsilon_stage,
                    "Dataset": dataset_name,
                    "Seed": seed,
                    "Strategy": strategy,
                    "Base_Strategy": base_strategy,
                    "Ablation_Variant": ablation_variant,
                    "Use_Temporal_Memory": int(strategy_defense_config.get("use_temporal_memory", True)),
                    "Use_Adaptive_Weighting": int(strategy_defense_config.get("use_adaptive_weighting", True)),
                    "Use_Circuit_Breaker": int(strategy_defense_config.get("use_circuit_breaker", True)),
                    "Trust_Threshold": TRUST_THRESHOLD,
                    "Round": stage_idx,
                    "Legacy_Stage_Equivalent": round_to_legacy_stage(stage_idx),
                    "Attack_Active": int(attack_active_stage),
                    "Client_Attack_Active": int(client_attack_state["attack_active"]),
                    "Client_Is_Attacker": int(client_attack_state["is_attacker"]),
                    "Client_Attack_Note": client_attack_state["attack_note"],
                    "Attack_Note": attack_note,
                    "Partition_Mode": partition_mode,
                    "Dirichlet_Alpha": partition_diag["Dirichlet_Alpha"],
                    "Client_Size_Min": partition_diag["Client_Size_Min"],
                    "Client_Size_Max": partition_diag["Client_Size_Max"],
                    "Label1_Proportion_Std": partition_diag["Label1_Proportion_Std"],
                    **drow
                })

            cb_round_diag = defense_diagnostics[0] if defense_diagnostics else {}

            g_weights += agg_update
            set_weights(g_model, g_weights, g_model)

            y_pred = g_model.predict(X_eval_sc)

            f1 = f1_score(
                y_eval,
                y_pred,
                average="macro"
            )

            # ASR de evasão: fração de amostras originalmente MALICIOSAS que,
            # após receberem o trigger, são classificadas como BENIGNAS.
            # Quanto menor, melhor a robustez da defesa.
            asr = (
                float(np.mean(g_model.predict(X_backdoor) == BACKDOOR_TARGET_CLASS))
                if len(X_backdoor) > 0
                else 0.0
            )

            strategy_asr_values.append(asr)
            if attack_active_stage:
                strategy_asr_active_values.append(asr)
            else:
                strategy_asr_inactive_values.append(asr)

            asr_mean_attack_active_so_far = (
                float(np.mean(strategy_asr_active_values))
                if strategy_asr_active_values
                else np.nan
            )
            asr_mean_attack_inactive_so_far = (
                float(np.mean(strategy_asr_inactive_values))
                if strategy_asr_inactive_values
                else np.nan
            )
            asr_peak_so_far = (
                float(np.max(strategy_asr_values))
                if strategy_asr_values
                else np.nan
            )

            stage_time = time.perf_counter() - stage_start_time
            cumulative_strategy_time = time.perf_counter() - strategy_start_time

            results.append({
                "Benchmark": benchmark_label,
                "Attack": attack_label,
                "Cumulative_Profile": cumulative_profile if attack_name == "Cumulative" else "",
                "Cumulative_Epsilon": cumulative_epsilon_stage,
                "Dataset": dataset_name,
                "Seed": seed,
                "Strategy": strategy,
                "Base_Strategy": base_strategy,
                "Ablation_Variant": ablation_variant,
                "Use_Temporal_Memory": int(strategy_defense_config.get("use_temporal_memory", True)),
                "Use_Adaptive_Weighting": int(strategy_defense_config.get("use_adaptive_weighting", True)),
                "Use_Circuit_Breaker": int(strategy_defense_config.get("use_circuit_breaker", True)),
                "Trust_Threshold": TRUST_THRESHOLD,
                "Round": stage_idx,
                "Round_Name": f"R{stage_idx}",
                "Stage": stage_idx,
                "Stage_Name": stage_name,
                "Attack_Active": int(attack_active_stage),
                "Attack_Note": attack_note,
                "Legacy_Stage_Equivalent": round_to_legacy_stage(stage_idx),
                "Active_Attackers": active_attackers_stage,
                    "Pressure_Fraction": max((x.get("pressure_fraction", 0.0) for x in client_attack_states), default=0.0),
                    "Compromised_Clients": max((x.get("compromised_clients", 0) for x in client_attack_states), default=0),
                "Total_Nodes": len(nodes),
                "Trusted_Clients": trusted_clients,
                "Circuit_Breaker": circuit_breaker,
                "Circuit_Breaker_Mode": strategy_defense_config.get("circuit_breaker_mode", "not_applicable") if base_strategy == "Adaptive_Ultimate" else "not_applicable",
                "CB_State": cb_round_diag.get("CB_State", "NOT_APPLICABLE"),
                "CB_Lambda": float(cb_round_diag.get("CB_Lambda", 1.0)),
                "CB_Trust_Mass": float(cb_round_diag.get("CB_Trust_Mass", np.nan)),
                "CB_Mean_Trusted_Trust": float(cb_round_diag.get("CB_Mean_Trusted_Trust", np.nan)),
                "CB_Effective_Clients": float(cb_round_diag.get("CB_Effective_Clients", np.nan)),
                "CB_Aggregate_Alignment": float(cb_round_diag.get("CB_Aggregate_Alignment", np.nan)),
                "CB_Min_Trusted": float(cb_round_diag.get("CB_Min_Trusted", np.nan)),
                "CB_Min_ESS": float(cb_round_diag.get("CB_Min_ESS", np.nan)),
                "CB_Calibrating": int(cb_round_diag.get("CB_Calibrating", 0)),
                "Partition_Mode": partition_mode,
                "Dirichlet_Alpha": partition_diag["Dirichlet_Alpha"],
                "Client_Size_Min": partition_diag["Client_Size_Min"],
                "Client_Size_Max": partition_diag["Client_Size_Max"],
                "Client_Size_Mean": partition_diag["Client_Size_Mean"],
                "Label1_Proportion_Std": partition_diag["Label1_Proportion_Std"],
                "F1": f1,
                "ASR": asr,
                "ASR_Clean_Trigger": clean_trigger_asr,
                "ASR_Clean_Source_Error": clean_source_error,
                "Delta_ASR_vs_CleanTrigger": asr - clean_trigger_asr,
                "Trigger_Feature_Index": trigger_spec["feature_idx"],
                "Trigger_Value": trigger_spec["value"],
                "Trigger_Quantile": trigger_spec["quantile"],
                "Trigger_Feature_Separation": trigger_spec["separation"],
                "ASR_Attack_Active": (
                    asr if attack_active_stage else np.nan
                ),
                "ASR_Attack_Inactive": (
                    asr if not attack_active_stage else np.nan
                ),
                "ASR_Mean_Attack_Active_SoFar": asr_mean_attack_active_so_far,
                "ASR_Mean_Attack_Inactive_SoFar": asr_mean_attack_inactive_so_far,
                "ASR_Peak_SoFar": asr_peak_so_far,
                "Time": stage_time,
                "Cumulative_Time_Strategy": cumulative_strategy_time
            })

        final = results[-1]

        print(
            f"      {strategy.ljust(17)} | "
            f"F1={final['F1']:.4f} | "
            f"ASR={final['ASR']:.4f} | "
            f"CleanASR={final['ASR_Clean_Trigger']:.4f} | "
            f"DeltaASR={final['Delta_ASR_vs_CleanTrigger']:.4f} | "
            f"Time={final['Time']:.4f}s | "
            f"CumTime={final['Cumulative_Time_Strategy']:.4f}s"
        )

    return results, client_logs


# ==============================================================================
# 6. SUMARIZAÇÃO E GRÁFICOS
# ==============================================================================

def save_outputs(full_df: pd.DataFrame, out_dir: Path):
    out_dir.mkdir(parents=True, exist_ok=True)
    plots_dir = out_dir / "plots"
    plots_dir.mkdir(exist_ok=True)

    full_df.to_csv(out_dir / "raw_results.csv", index=False)

    # Comparação direta do comportamento do Circuit Breaker: adaptativo final vs fixo legado vs sem CB.
    cb_compare_strategies = [
        "Adaptive_Ultimate",
        "Adaptive_FixedCircuitBreaker",
        "Adaptive_NoCircuitBreaker",
    ]
    cb_df = full_df[full_df["Strategy"].isin(cb_compare_strategies)].copy()
    if not cb_df.empty:
        cb_df["CB_Degraded"] = (cb_df["CB_State"].astype(str) == "DEGRADED").astype(int)
        cb_df["CB_Break"] = (cb_df["CB_State"].astype(str).isin(["BREAK", "FIXED_BREAK"])).astype(int)
        cb_summary = (
            cb_df.groupby("Strategy", as_index=False)
            .agg(
                F1_Mean=("F1", "mean"),
                ASR_Mean=("ASR", "mean"),
                ASR_Attack_Active_Mean=("ASR_Attack_Active", "mean"),
                Circuit_Breaker_Rate=("Circuit_Breaker", "mean"),
                CB_Degraded_Rate=("CB_Degraded", "mean"),
                CB_Break_Rate=("CB_Break", "mean"),
                CB_Lambda_Mean=("CB_Lambda", "mean"),
                Trusted_Clients_Mean=("Trusted_Clients", "mean"),
            )
        )
        cb_summary.to_csv(out_dir / "circuit_breaker_comparison.csv", index=False)

    adaptive_ablation_df = full_df[
        full_df["Strategy"].isin(ADAPTIVE_VARIANT_TO_CONFIG.keys())
    ].copy()
    if not adaptive_ablation_df.empty:
        adaptive_ablation_df.to_csv(out_dir / "ablation_raw_results.csv", index=False)

    summary_by_stage = (
        full_df
        .groupby(["Benchmark", "Attack", "Dataset", "Partition_Mode", "Strategy", "Stage"], as_index=False)
        .agg(
            F1_Mean=("F1", "mean"),
            F1_Std=("F1", "std"),
            ASR_Mean=("ASR", "mean"),
            ASR_Std=("ASR", "std"),
            Time_Mean=("Time", "mean"),
            Time_Std=("Time", "std"),
            Cumulative_Time_Strategy_Mean=("Cumulative_Time_Strategy", "mean"),
            Cumulative_Time_Strategy_Std=("Cumulative_Time_Strategy", "std"),
            Attack_Active=("Attack_Active", "max"),
            Active_Attackers=("Active_Attackers", "max"),
            Trusted_Clients_Mean=("Trusted_Clients", "mean"),
            Circuit_Breaker_Rate=("Circuit_Breaker", "mean")
        )
    )

    for c in ["F1_Std", "ASR_Std", "Time_Std", "Cumulative_Time_Strategy_Std"]:
        summary_by_stage[c] = summary_by_stage[c].fillna(0)

    summary_by_stage.to_csv(out_dir / "summary_by_stage.csv", index=False)

    summary_by_strategy = (
        full_df
        .groupby(["Benchmark", "Attack", "Dataset", "Partition_Mode", "Strategy"], as_index=False)
        .agg(
            F1_Global_Mean=("F1", "mean"),
            F1_Global_Std=("F1", "std"),
            ASR_Global_Mean=("ASR", "mean"),
            ASR_Global_Std=("ASR", "std"),
            Time_Global_Mean=("Time", "mean"),
            Time_Global_Std=("Time", "std"),
            Total_Time_Strategy_Max=("Cumulative_Time_Strategy", "max"),
            Circuit_Breaker_Rate=("Circuit_Breaker", "mean")
        )
    )

    summary_by_strategy.to_csv(out_dir / "summary_by_strategy.csv", index=False)

    final_rows = []
    for (_, _, _, _, strategy), g in summary_by_stage.groupby(["Attack", "Dataset", "Benchmark", "Partition_Mode", "Strategy"]):
        max_stage = g["Stage"].max()
        final_rows.append(g[g["Stage"] == max_stage])

    final_stage = pd.concat(final_rows, ignore_index=True)

    final_stage = final_stage.sort_values(
        ["Attack", "Dataset", "ASR_Mean", "F1_Mean"],
        ascending=[True, True, True, False]
    )

    final_stage.to_csv(out_dir / "final_stage_ranking.csv", index=False)

    # Plots
    for metric, std_col, ylabel, ylim in [
        ("F1_Mean", "F1_Std", "F1-score", (-0.02, 1.02)),
        ("ASR_Mean", "ASR_Std", "ASR", (-0.02, 1.02)),
        ("Time_Mean", "Time_Std", "Time (seconds)", None)
    ]:
        plt.figure(figsize=(10, 5), dpi=180)

        for strategy, g in summary_by_stage.groupby("Strategy"):
            g = g.sort_values("Stage")
            x = g["Stage"].values
            y = g[metric].values
            std = g[std_col].fillna(0).values

            plt.plot(
                x,
                y,
                marker="o",
                linewidth=1.8,
                markersize=3,
                label=strategy
            )

            plt.fill_between(
                x,
                y - std,
                y + std,
                alpha=0.12
            )

        attack = full_df["Attack"].iloc[0]
        dataset = full_df["Dataset"].iloc[0]

        plt.title(f"{attack} — {dataset}\nMean {ylabel} over stages")
        plt.xlabel("Stage")
        plt.ylabel(ylabel)
        plt.grid(True, alpha=0.25)
        plt.xticks(sorted(summary_by_stage["Stage"].unique()))

        if ylim:
            plt.ylim(*ylim)

        plt.legend(
            loc="center left",
            bbox_to_anchor=(1.02, 0.5),
            frameon=False,
            fontsize=9
        )

        plt.tight_layout()
        plt.savefig(plots_dir / f"{metric.replace('_Mean', '')}_over_stage.png", bbox_inches="tight")
        plt.close()

    return summary_by_stage, summary_by_strategy, final_stage


def save_global_outputs(all_raw: pd.DataFrame):
    OUTPUT_ROOT.mkdir(exist_ok=True)

    all_raw.to_csv(OUTPUT_ROOT / "all_raw_results.csv", index=False)

    all_stage = (
        all_raw
        .groupby(["Benchmark", "Attack", "Dataset", "Partition_Mode", "Strategy", "Stage"], as_index=False)
        .agg(
            F1_Mean=("F1", "mean"),
            F1_Std=("F1", "std"),
            ASR_Mean=("ASR", "mean"),
            ASR_Std=("ASR", "std"),
            Time_Mean=("Time", "mean"),
            Time_Std=("Time", "std"),
            Attack_Active=("Attack_Active", "max"),
            Active_Attackers=("Active_Attackers", "max"),
            Trusted_Clients_Mean=("Trusted_Clients", "mean"),
            Circuit_Breaker_Rate=("Circuit_Breaker", "mean")
        )
    )

    all_stage.to_csv(OUTPUT_ROOT / "all_summary_by_stage.csv", index=False)

    final_rows = []

    for (attack, dataset, partition_mode, strategy), g in all_stage.groupby(["Attack", "Dataset", "Partition_Mode", "Strategy"]):
        max_stage = g["Stage"].max()
        final_rows.append(g[g["Stage"] == max_stage])

    all_final = pd.concat(final_rows, ignore_index=True)

    all_final = all_final.sort_values(
        ["Attack", "Dataset", "ASR_Mean", "F1_Mean"],
        ascending=[True, True, True, False]
    )

    all_final.to_csv(OUTPUT_ROOT / "all_final_stage_ranking.csv", index=False)

    diagnostic = (
        all_raw
        .groupby(["Attack", "Dataset"], as_index=False)
        .agg(
            rows=("F1", "count"),
            seeds=("Seed", "nunique"),
            strategies=("Strategy", "nunique"),
            min_stage=("Stage", "min"),
            max_stage=("Stage", "max"),
            total_nodes=("Total_Nodes", "max"),
            mean_time=("Time", "mean"),
            total_time=("Time", "sum")
        )
    )

    diagnostic.to_csv(OUTPUT_ROOT / "all_diagnostic.csv", index=False)

    return all_stage, all_final, diagnostic



# ==============================================================================
# 7. ANÁLISE ESTATÍSTICA PÓS-BANCA
# ==============================================================================
# Unidade experimental: execução/seed dentro de cada combinação
# Dataset x Ataque x Partition_Mode. Os rounds NÃO são tratados como amostras
# independentes.
#
# Métricas por seed:
#   - F1_Mean_Over_Rounds / ASR_Mean_Over_Rounds
#   - F1_Final_Round / ASR_Final_Round
#
# Procedimento:
#   1) estatística descritiva + IC95% com distribuição t;
#   2) Friedman para comparação global entre métodos;
#   3) Wilcoxon pareado entre métodos + correção de Holm;
#   4) tamanho de efeito: Cliff's delta e Cohen dz pareado.
# ==============================================================================

STATISTICAL_ALPHA = 0.05


def _ci95(values):
    """IC95% da média usando distribuição t; retorna (mean, std, low, high, n)."""
    x = pd.Series(values).dropna().astype(float).values
    n = len(x)

    if n == 0:
        return np.nan, np.nan, np.nan, np.nan, 0

    mean = float(np.mean(x))
    std = float(np.std(x, ddof=1)) if n > 1 else 0.0

    if n < 2:
        return mean, std, mean, mean, n

    sem = stats.sem(x, nan_policy="omit")
    tcrit = stats.t.ppf(0.975, df=n - 1)
    margin = float(tcrit * sem)

    return mean, std, mean - margin, mean + margin, n


def _holm_adjust(p_values):
    """Correção de Holm step-down, sem dependência de statsmodels."""
    p = np.asarray(p_values, dtype=float)
    m = len(p)

    if m == 0:
        return np.array([], dtype=float)

    order = np.argsort(p)
    adjusted_sorted = np.empty(m, dtype=float)
    running_max = 0.0

    for rank, idx in enumerate(order):
        factor = m - rank
        candidate = min(1.0, factor * p[idx])
        running_max = max(running_max, candidate)
        adjusted_sorted[rank] = running_max

    adjusted = np.empty(m, dtype=float)
    for rank, idx in enumerate(order):
        adjusted[idx] = adjusted_sorted[rank]

    return adjusted


def _cliffs_delta(x, y):
    """Cliff's delta: efeito ordinal entre duas amostras."""
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    x = x[np.isfinite(x)]
    y = y[np.isfinite(y)]

    if len(x) == 0 or len(y) == 0:
        return np.nan

    greater = 0
    lower = 0

    for xv in x:
        greater += np.sum(xv > y)
        lower += np.sum(xv < y)

    return float((greater - lower) / (len(x) * len(y)))


def _cohen_dz(x, y):
    """Cohen dz para medidas pareadas."""
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    mask = np.isfinite(x) & np.isfinite(y)
    d = x[mask] - y[mask]

    if len(d) < 2:
        return np.nan

    sd = np.std(d, ddof=1)

    if sd == 0:
        return 0.0 if np.mean(d) == 0 else np.sign(np.mean(d)) * np.inf

    return float(np.mean(d) / sd)


def build_seed_level_metrics(all_raw: pd.DataFrame) -> pd.DataFrame:
    """
    Reduz cada execução/seed a uma única observação por método e cenário,
    evitando pseudo-replicação por round.
    """
    group_cols = [
        "Dataset",
        "Attack",
        "Partition_Mode",
        "Seed",
        "Strategy"
    ]

    seed_mean = (
        all_raw
        .groupby(group_cols, as_index=False)
        .agg(
            F1_Mean_Over_Rounds=("F1", "mean"),
            ASR_Mean_Over_Rounds=("ASR", "mean"),
            Time_Mean_Over_Rounds=("Time", "mean"),
            Circuit_Breaker_Rate=("Circuit_Breaker", "mean")
        )
    )

    final_idx = (
        all_raw
        .groupby(group_cols)["Round"]
        .idxmax()
    )

    final_rows = (
        all_raw
        .loc[final_idx, group_cols + ["F1", "ASR", "Time"]]
        .rename(columns={
            "F1": "F1_Final_Round",
            "ASR": "ASR_Final_Round",
            "Time": "Time_Final_Round"
        })
    )

    return seed_mean.merge(final_rows, on=group_cols, how="left")


def descriptive_ci95(seed_metrics: pd.DataFrame) -> pd.DataFrame:
    metric_cols = [
        "F1_Mean_Over_Rounds",
        "ASR_Mean_Over_Rounds",
        "F1_Final_Round",
        "ASR_Final_Round",
        "Time_Mean_Over_Rounds",
        "Circuit_Breaker_Rate"
    ]

    rows = []

    for keys, g in seed_metrics.groupby(
        ["Dataset", "Attack", "Partition_Mode", "Strategy"],
        dropna=False
    ):
        dataset, attack, partition_mode, strategy = keys

        for metric in metric_cols:
            mean, std, low, high, n = _ci95(g[metric])

            rows.append({
                "Dataset": dataset,
                "Attack": attack,
                "Partition_Mode": partition_mode,
                "Strategy": strategy,
                "Metric": metric,
                "N_Seeds": n,
                "Mean": mean,
                "Std": std,
                "CI95_Low": low,
                "CI95_High": high,
                "CI95_Width": high - low if np.isfinite(low) and np.isfinite(high) else np.nan
            })

    return pd.DataFrame(rows)


def friedman_tests(seed_metrics: pd.DataFrame) -> pd.DataFrame:
    """
    Friedman sobre seeds pareadas. Cada linha da matriz = uma seed;
    cada coluna = um método.
    """
    test_metrics = [
        "F1_Mean_Over_Rounds",
        "ASR_Mean_Over_Rounds",
        "F1_Final_Round",
        "ASR_Final_Round"
    ]

    rows = []

    for keys, g in seed_metrics.groupby(
        ["Dataset", "Attack", "Partition_Mode"],
        dropna=False
    ):
        dataset, attack, partition_mode = keys

        for metric in test_metrics:
            pivot = g.pivot(index="Seed", columns="Strategy", values=metric).dropna(axis=0)

            n_seeds = len(pivot)
            n_methods = pivot.shape[1]

            if n_seeds < 2 or n_methods < 3:
                rows.append({
                    "Dataset": dataset,
                    "Attack": attack,
                    "Partition_Mode": partition_mode,
                    "Metric": metric,
                    "N_Seeds_Paired": n_seeds,
                    "N_Methods": n_methods,
                    "Friedman_Statistic": np.nan,
                    "P_Value": np.nan,
                    "Significant_0_05": False,
                    "Status": "insufficient_paired_data"
                })
                continue

            arrays = [pivot[col].values for col in pivot.columns]

            try:
                statistic, pvalue = stats.friedmanchisquare(*arrays)
                status = "ok"
            except ValueError:
                statistic, pvalue = np.nan, np.nan
                status = "constant_or_invalid_data"

            rows.append({
                "Dataset": dataset,
                "Attack": attack,
                "Partition_Mode": partition_mode,
                "Metric": metric,
                "N_Seeds_Paired": n_seeds,
                "N_Methods": n_methods,
                "Friedman_Statistic": statistic,
                "P_Value": pvalue,
                "Significant_0_05": bool(np.isfinite(pvalue) and pvalue < STATISTICAL_ALPHA),
                "Status": status
            })

    return pd.DataFrame(rows)


def pairwise_wilcoxon_holm(seed_metrics: pd.DataFrame) -> pd.DataFrame:
    """
    Wilcoxon signed-rank pareado pelas mesmas seeds.
    A correção Holm é aplicada separadamente por
    Dataset x Attack x Partition_Mode x Metric.
    """
    test_metrics = [
        "F1_Mean_Over_Rounds",
        "ASR_Mean_Over_Rounds",
        "F1_Final_Round",
        "ASR_Final_Round"
    ]

    all_rows = []

    for keys, g in seed_metrics.groupby(
        ["Dataset", "Attack", "Partition_Mode"],
        dropna=False
    ):
        dataset, attack, partition_mode = keys

        for metric in test_metrics:
            pivot = g.pivot(index="Seed", columns="Strategy", values=metric)
            methods = sorted(pivot.columns.tolist())
            local_rows = []

            for method_a, method_b in combinations(methods, 2):
                paired = pivot[[method_a, method_b]].dropna()

                x = paired[method_a].values.astype(float)
                y = paired[method_b].values.astype(float)
                n = len(paired)

                if n < 2:
                    statistic = np.nan
                    pvalue = np.nan
                    status = "insufficient_paired_data"
                elif np.allclose(x, y, equal_nan=True):
                    statistic = 0.0
                    pvalue = 1.0
                    status = "identical_values"
                else:
                    try:
                        statistic, pvalue = stats.wilcoxon(
                            x,
                            y,
                            zero_method="wilcox",
                            alternative="two-sided",
                            method="auto"
                        )
                        status = "ok"
                    except ValueError:
                        statistic = np.nan
                        pvalue = np.nan
                        status = "invalid_data"

                local_rows.append({
                    "Dataset": dataset,
                    "Attack": attack,
                    "Partition_Mode": partition_mode,
                    "Metric": metric,
                    "Method_A": method_a,
                    "Method_B": method_b,
                    "N_Seeds_Paired": n,
                    "Mean_A": float(np.mean(x)) if n else np.nan,
                    "Mean_B": float(np.mean(y)) if n else np.nan,
                    "Mean_Difference_A_minus_B": float(np.mean(x - y)) if n else np.nan,
                    "Wilcoxon_Statistic": statistic,
                    "P_Value_Raw": pvalue,
                    "Cliffs_Delta_A_vs_B": _cliffs_delta(x, y),
                    "Cohen_dz_A_minus_B": _cohen_dz(x, y),
                    "Status": status
                })

            if local_rows:
                valid_p = [
                    row["P_Value_Raw"] if np.isfinite(row["P_Value_Raw"]) else 1.0
                    for row in local_rows
                ]
                adjusted = _holm_adjust(valid_p)

                for row, p_adj in zip(local_rows, adjusted):
                    row["P_Value_Holm"] = float(p_adj)
                    row["Significant_Holm_0_05"] = bool(p_adj < STATISTICAL_ALPHA)

                all_rows.extend(local_rows)

    return pd.DataFrame(all_rows)


def save_statistical_analysis(all_raw: pd.DataFrame):
    stats_dir = OUTPUT_ROOT / "statistical_analysis"
    stats_dir.mkdir(parents=True, exist_ok=True)

    seed_metrics = build_seed_level_metrics(all_raw)
    descriptive = descriptive_ci95(seed_metrics)
    friedman = friedman_tests(seed_metrics)
    pairwise = pairwise_wilcoxon_holm(seed_metrics)

    seed_metrics.to_csv(stats_dir / "metrics_by_seed.csv", index=False)
    descriptive.to_csv(stats_dir / "descriptive_ci95.csv", index=False)
    friedman.to_csv(stats_dir / "friedman_tests.csv", index=False)
    pairwise.to_csv(stats_dir / "wilcoxon_holm_effect_sizes.csv", index=False)

    # Tabela compacta para uso direto na tese
    thesis_summary = (
        descriptive[
            descriptive["Metric"].isin([
                "F1_Mean_Over_Rounds",
                "ASR_Mean_Over_Rounds",
                "F1_Final_Round",
                "ASR_Final_Round"
            ])
        ]
        .copy()
    )

    thesis_summary["Mean_CI95"] = thesis_summary.apply(
        lambda r: (
            f"{r['Mean']:.4f} "
            f"[{r['CI95_Low']:.4f}, {r['CI95_High']:.4f}]"
            if np.isfinite(r["Mean"]) else ""
        ),
        axis=1
    )

    thesis_summary.to_csv(stats_dir / "thesis_summary_ci95.csv", index=False)

    return {
        "seed_metrics": seed_metrics,
        "descriptive": descriptive,
        "friedman": friedman,
        "pairwise": pairwise,
        "thesis_summary": thesis_summary
    }


# ==============================================================================
# 7B. ANÁLISE TEMPORAL DA DEFESA E DO CIRCUIT BREAKER
# ==============================================================================

def analyze_temporal_defense(client_log_df: pd.DataFrame):
    if client_log_df is None or client_log_df.empty:
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()
    logs = client_log_df[client_log_df["Strategy"].isin([
        "Adaptive_Ultimate",
        "Adaptive_FixedCircuitBreaker",
    ])].copy()
    episode_rows, cb_rows, summary_rows = [], [], []
    group_cols = ["Dataset", "Attack", "Partition_Mode", "Seed", "Strategy"]
    for keys, g in logs.groupby(group_cols, dropna=False):
        meta = dict(zip(group_cols, keys))
        g = g.sort_values(["Round", "Client"]).copy()

        # Ground truth de detecção é CLIENTE-ESPECÍFICO.
        # Attack_Active representa apenas se existe algum ataque ativo na rodada e
        # NÃO deve rotular todos os clientes daquela rodada como maliciosos.
        g["Predicted_Malicious"] = (g["Client_Trusted"].astype(int) == 0).astype(int)
        y = g["Client_Is_Attacker"].astype(int).to_numpy()
        p = g["Predicted_Malicious"].to_numpy()
        tp = int(((y == 1) & (p == 1)).sum())
        fp = int(((y == 0) & (p == 1)).sum())
        tn = int(((y == 0) & (p == 0)).sum())
        fn = int(((y == 1) & (p == 0)).sum())
        delays=[]; rec_delays=[]; total_eps=det_eps=0
        for client,cg in g.groupby("Client"):
            cg=cg.sort_values("Round").reset_index(drop=True); st=cg["Client_Is_Attacker"].astype(int).to_numpy(); rr=cg["Round"].astype(int).to_numpy(); start=None; eps=[]
            for j,v in enumerate(st):
                if v==1 and start is None: start=rr[j]
                last=j==len(st)-1
                if start is not None and (v==0 or last):
                    end=rr[j-1] if v==0 else rr[j]; nxt=rr[j] if v==0 else None; eps.append((start,end,nxt)); start=None
            for epn,(start,end,nxt) in enumerate(eps,1):
                total_eps+=1; during=cg[(cg.Round>=start)&(cg.Round<=end)]; det=during[during.Client_Trusted.astype(int)==0]
                dr=int(det.Round.iloc[0]) if len(det) else np.nan; dd=dr-start if len(det) else np.nan
                if len(det): det_eps+=1; delays.append(dd)
                rec=np.nan; rd=np.nan
                if nxt is not None:
                    recs=cg[(cg.Round>=nxt)&(cg.Client_Trusted.astype(int)==1)]
                    if len(recs): rec=int(recs.Round.iloc[0]); rd=rec-nxt; rec_delays.append(rd)
                episode_rows.append({**meta,"Client":client,"Attack_Episode":epn,"Attack_Start_Round":start,"Attack_End_Round":end,"Detection_Round":dr,"Detection_Delay_Rounds":dd,"Recovery_Round":rec,"Recovery_Delay_Rounds":rd,"Detected":int(len(det)>0)})
        rg=g.groupby("Round",as_index=False).agg(
            Attack_Active=("Attack_Active","max"),
            Circuit_Breaker=("Circuit_Breaker","max"),
            Trusted_Clients=("Trusted_Clients","max"),
            Majority_Threshold=("Majority_Threshold","max"),
            CB_State=("CB_State","first"),
            CB_Lambda=("CB_Lambda","first"),
            Circuit_Breaker_Reason=("Circuit_Breaker_Reason","first"),
        ).sort_values("Round")
        rg["CB_Start"]=((rg.Circuit_Breaker==1)&(rg.Circuit_Breaker.shift(fill_value=0)==0)).astype(int); rg["CB_End"]=((rg.Circuit_Breaker==0)&(rg.Circuit_Breaker.shift(fill_value=0)==1)).astype(int)
        onset=rg.loc[rg.Attack_Active==1,"Round"]; onset=int(onset.min()) if len(onset) else np.nan
        cbd=rg[(rg.Circuit_Breaker==1)&(rg.Attack_Active==1)]; first=int(cbd.Round.min()) if len(cbd) else np.nan
        for _,r in rg[rg.CB_Start==1].iterrows(): cb_rows.append({**meta,"Event":"CB_TRIGGER","Round":int(r.Round),"Attack_Active":int(r.Attack_Active),"Trusted_Clients":int(r.Trusted_Clients),"Majority_Threshold":int(r.Majority_Threshold),"NoAttack_Activation":int(r.Attack_Active==0),"False_Trigger":int(r.Attack_Active==0),"Reason":str(r.get("Circuit_Breaker_Reason","unknown")),"CB_State":str(r.get("CB_State","unknown")),"CB_Lambda":float(r.get("CB_Lambda",np.nan))})
        for _,r in rg[rg.CB_End==1].iterrows(): cb_rows.append({**meta,"Event":"CB_RELEASE","Round":int(r.Round),"Attack_Active":int(r.Attack_Active),"Trusted_Clients":int(r.Trusted_Clients),"Majority_Threshold":int(r.Majority_Threshold),"NoAttack_Activation":0,"False_Trigger":0,"Reason":"circuit_breaker_release","CB_State":str(r.get("CB_State","unknown")),"CB_Lambda":float(r.get("CB_Lambda",np.nan))})
        no_attack_cb_rounds = int(((rg.Circuit_Breaker == 1) & (rg.Attack_Active == 0)).sum())
        summary_rows.append({
            **meta,
            "Ground_Truth_Field": "Client_Is_Attacker",
            "Trust_Threshold": TRUST_THRESHOLD,
            # nomes explícitos
            "TP_Malicious_Rejected": tp,
            "FP_Honest_Rejected": fp,
            "TN_Honest_Accepted": tn,
            "FN_Malicious_Accepted": fn,
            "Malicious_Detection_Precision": tp/(tp+fp) if tp+fp else np.nan,
            "Malicious_Detection_Recall": tp/(tp+fn) if tp+fn else np.nan,
            "Honest_Rejection_Rate": fp/(fp+tn) if fp+tn else np.nan,
            # aliases legados para rastreabilidade
            "TP": tp, "FP": fp, "TN": tn, "FN": fn,
            "Detection_Precision": tp/(tp+fp) if tp+fp else np.nan,
            "Detection_Recall": tp/(tp+fn) if tp+fn else np.nan,
            "False_Positive_Rate": fp/(fp+tn) if fp+tn else np.nan,
            "Attack_Episodes": total_eps,
            "Detected_Episodes": det_eps,
            "Episode_Detection_Rate": det_eps/total_eps if total_eps else np.nan,
            "Detection_Delay_Mean": float(np.mean(delays)) if delays else np.nan,
            "Recovery_Delay_Mean": float(np.mean(rec_delays)) if rec_delays else np.nan,
            "CB_Trigger_Count": int(rg.CB_Start.sum()),
            "CB_Active_Rounds": int(rg.Circuit_Breaker.sum()),
            "CB_NoAttack_Active_Rounds": no_attack_cb_rounds,
            "CB_False_Trigger_Rounds": no_attack_cb_rounds,  # alias legado; interpretar como ativação sem ataque observado
            "CB_First_Trigger_Round": first,
            "CB_Trigger_Delay_From_Attack_Onset": first-onset if not pd.isna(first) and not pd.isna(onset) else np.nan,
        })
    return pd.DataFrame(episode_rows), pd.DataFrame(cb_rows), pd.DataFrame(summary_rows)

def save_temporal_defense_analysis(client_log_df: pd.DataFrame, out_dir: Path):
    episodes, cb_events, summary = analyze_temporal_defense(client_log_df)
    if not episodes.empty: episodes.to_csv(out_dir / "client_detection_recovery.csv", index=False)
    if not cb_events.empty: cb_events.to_csv(out_dir / "circuit_breaker_events.csv", index=False)
    if not summary.empty: summary.to_csv(out_dir / "temporal_defense_summary.csv", index=False)
    return episodes, cb_events, summary

# ==============================================================================
# 8. MAIN
# ==============================================================================


# =====================================================================
# VALIDACAO AUTOMATICA DO SMOKE TEST
# =====================================================================

def _validation_row(check, passed, details="", severity="ERROR"):
    return {
        "Check": str(check),
        "Passed": bool(passed),
        "Severity": str(severity),
        "Details": str(details),
    }


def validate_smoke_test_outputs(
    raw_df,
    adaptive_client_log_df=None,
    expected_rounds=N_FEDERATED_ROUNDS,
    expected_strategies=None,
):
    """
    Valida uma execução piloto antes da campanha completa.

    Unidade esperada: um cenário Dataset x Attack x Partition_Mode.
    A função não usa desempenho preditivo para aprovar/reprovar; ela testa
    consistência estrutural e comportamento esperado das variantes.
    """
    checks = []

    if expected_strategies is None:
        expected_strategies = list(STRATEGIES)

    if raw_df is None or len(raw_df) == 0:
        return pd.DataFrame([
            _validation_row("raw_results_nonempty", False, "raw_results vazio")
        ])

    df = raw_df.copy()

    # 1. Estratégias presentes.
    present = sorted(df["Strategy"].dropna().astype(str).unique().tolist()) if "Strategy" in df.columns else []
    missing = sorted(set(expected_strategies) - set(present))
    checks.append(_validation_row(
        "all_expected_strategies_present",
        len(missing) == 0,
        f"present={present}; missing={missing}"
    ))

    # 2. Número de rounds por estratégia/seed.
    if {"Strategy", "Seed", "Round"}.issubset(df.columns):
        grp = df.groupby(["Strategy", "Seed"])["Round"].nunique()
        bad = grp[grp != expected_rounds]
        checks.append(_validation_row(
            "expected_round_count_per_strategy_seed",
            len(bad) == 0,
            "OK" if len(bad) == 0 else bad.to_dict()
        ))
    else:
        checks.append(_validation_row(
            "expected_round_count_per_strategy_seed",
            False,
            "Colunas Strategy/Seed/Round ausentes"
        ))

    # 3. Cobertura exata 0..expected_rounds-1.
    if "Round" in df.columns:
        rounds = sorted(pd.Series(df["Round"]).dropna().astype(int).unique().tolist())
        expected = list(range(expected_rounds))
        checks.append(_validation_row(
            "round_index_coverage",
            rounds == expected,
            f"min={min(rounds) if rounds else None}; max={max(rounds) if rounds else None}; n={len(rounds)}"
        ))

    # 4. NaN/Inf em métricas numéricas críticas.
    numeric_candidates = [
        "F1", "ASR", "Time", "Trusted_Clients", "Circuit_Breaker",
        "Legacy_Stage_Equivalent"
    ]
    numeric_cols = [c for c in numeric_candidates if c in df.columns]
    invalid_numeric = {}
    for c in numeric_cols:
        vals = pd.to_numeric(df[c], errors="coerce")
        invalid = int(vals.isna().sum() + np.isinf(vals.to_numpy(dtype=float, na_value=np.nan)).sum())
        if invalid:
            invalid_numeric[c] = invalid
    checks.append(_validation_row(
        "no_nan_or_inf_in_critical_metrics",
        len(invalid_numeric) == 0,
        "OK" if not invalid_numeric else invalid_numeric
    ))

    # 5. Rótulos de ablação.
    expected_ablation = {
        "Adaptive_Ultimate": "Full",
        "Adaptive_FixedCircuitBreaker": "FixedCircuitBreaker",
        "Adaptive_NoTemporalMemory": "NoTemporalMemory",
        "Adaptive_NoAdaptiveWeighting": "NoAdaptiveWeighting",
        "Adaptive_NoCircuitBreaker": "NoCircuitBreaker",
    }
    bad_labels = []
    if {"Strategy", "Ablation_Variant"}.issubset(df.columns):
        for strategy, variant in expected_ablation.items():
            vals = set(df.loc[df["Strategy"] == strategy, "Ablation_Variant"].dropna().astype(str))
            if vals != {variant}:
                bad_labels.append((strategy, sorted(vals), variant))
        checks.append(_validation_row(
            "ablation_labels_correct",
            len(bad_labels) == 0,
            "OK" if not bad_labels else bad_labels
        ))
    else:
        checks.append(_validation_row(
            "ablation_labels_correct",
            False,
            "Strategy/Ablation_Variant ausentes"
        ))

    # 6. Flags das variantes.
    expected_flags = {
        "Adaptive_Ultimate": (1, 1, 1),
        "Adaptive_FixedCircuitBreaker": (1, 1, 1),
        "Adaptive_NoTemporalMemory": (0, 1, 1),
        "Adaptive_NoAdaptiveWeighting": (1, 0, 1),
        "Adaptive_NoCircuitBreaker": (1, 1, 0),
    }
    flag_cols = ["Use_Temporal_Memory", "Use_Adaptive_Weighting", "Use_Circuit_Breaker"]
    bad_flags = []
    if {"Strategy", *flag_cols}.issubset(df.columns):
        for strategy, expected_tuple in expected_flags.items():
            sub = df[df["Strategy"] == strategy]
            observed = set(
                tuple(int(v) for v in row)
                for row in sub[flag_cols].drop_duplicates().to_numpy()
            )
            if observed != {expected_tuple}:
                bad_flags.append((strategy, observed, expected_tuple))
        checks.append(_validation_row(
            "ablation_flags_correct",
            len(bad_flags) == 0,
            "OK" if not bad_flags else bad_flags
        ))
    else:
        checks.append(_validation_row(
            "ablation_flags_correct",
            False,
            "Flags de ablação ausentes"
        ))

    # 7. NoCircuitBreaker nunca pode registrar acionamento.
    if {"Strategy", "Circuit_Breaker"}.issubset(df.columns):
        sub = df[df["Strategy"] == "Adaptive_NoCircuitBreaker"]
        max_cb = pd.to_numeric(sub["Circuit_Breaker"], errors="coerce").fillna(0).max() if len(sub) else np.nan
        checks.append(_validation_row(
            "no_circuit_breaker_variant_never_triggers_cb",
            bool(len(sub) and max_cb == 0),
            f"max Circuit_Breaker={max_cb}"
        ))

    # 7B. Circuit Breaker adaptativo: lambda deve permanecer em [0,1] e
    # estados devem pertencer ao conjunto previsto. A presença de DEGRADED é
    # informativa (WARNING), pois pode não ocorrer em todo cenário.
    if {"Strategy", "CB_Lambda", "CB_State"}.issubset(df.columns):
        sub = df[df["Strategy"] == "Adaptive_Ultimate"].copy()
        if len(sub):
            lam = pd.to_numeric(sub["CB_Lambda"], errors="coerce")
            lambda_ok = bool(lam.notna().all() and ((lam >= 0.0) & (lam <= 1.0)).all())
            checks.append(_validation_row(
                "adaptive_cb_lambda_in_unit_interval",
                lambda_ok,
                f"min={lam.min()}, max={lam.max()}"
            ))
            allowed_states = {"CALIBRATION", "NORMAL", "DEGRADED", "BREAK"}
            observed_states = set(sub["CB_State"].dropna().astype(str).unique())
            checks.append(_validation_row(
                "adaptive_cb_states_valid",
                observed_states.issubset(allowed_states),
                f"observed={sorted(observed_states)}"
            ))
            checks.append(_validation_row(
                "adaptive_cb_degraded_mode_exercised",
                "DEGRADED" in observed_states,
                f"observed={sorted(observed_states)}",
                severity="WARNING"
            ))

        full = df[df["Strategy"] == "Adaptive_Ultimate"].copy()
        if len(full) and "Circuit_Breaker_Mode" in full.columns:
            observed_full_modes = set(full["Circuit_Breaker_Mode"].dropna().astype(str).unique())
            checks.append(_validation_row(
                "full_csai4iot_uses_adaptive_cb_mode",
                observed_full_modes == {"adaptive"},
                f"observed={sorted(observed_full_modes)}"
            ))

        fixed = df[df["Strategy"] == "Adaptive_FixedCircuitBreaker"].copy()
        if len(fixed) and "Circuit_Breaker_Mode" in fixed.columns:
            observed_fixed_modes = set(fixed["Circuit_Breaker_Mode"].dropna().astype(str).unique())
            checks.append(_validation_row(
                "fixed_cb_control_uses_fixed_mode",
                observed_fixed_modes == {"fixed"},
                f"observed={sorted(observed_fixed_modes)}"
            ))

    # 8. Legacy stage deve cobrir 0..16 e ser monotônico.
    if {"Round", "Legacy_Stage_Equivalent"}.issubset(df.columns):
        tmp = (
            df[["Round", "Legacy_Stage_Equivalent"]]
            .drop_duplicates()
            .sort_values("Round")
        )
        legacy = pd.to_numeric(tmp["Legacy_Stage_Equivalent"], errors="coerce")
        monotonic = legacy.is_monotonic_increasing
        covers = (legacy.min() == 0 and legacy.max() == LEGACY_MAX_STAGE)
        checks.append(_validation_row(
            "legacy_stage_mapping_valid",
            bool(monotonic and covers),
            f"min={legacy.min()}, max={legacy.max()}, monotonic={monotonic}"
        ))

    # 9. Logs adaptativos por cliente.
    if adaptive_client_log_df is not None and len(adaptive_client_log_df):
        log = adaptive_client_log_df.copy()
        required = {
            "Strategy", "Seed", "Round", "Client",
            "Trust_Before", "Trust_After", "Adaptive_Weight",
            "Client_Trusted", "Client_Attack_Active", "Client_Is_Attacker",
            "Trust_Threshold", "Circuit_Breaker"
        }
        missing_cols = sorted(required - set(log.columns))
        checks.append(_validation_row(
            "adaptive_client_log_required_columns",
            len(missing_cols) == 0,
            "OK" if not missing_cols else f"missing={missing_cols}"
        ))

        # 9B. O threshold efetivamente persistido no log deve ser o configurado.
        if "Trust_Threshold" in log.columns:
            tau_vals = pd.to_numeric(log["Trust_Threshold"], errors="coerce").dropna().unique()
            tau_ok = bool(len(tau_vals) == 1 and np.isclose(float(tau_vals[0]), TRUST_THRESHOLD))
            checks.append(_validation_row(
                "trust_threshold_logged_matches_config",
                tau_ok,
                f"observed={tau_vals.tolist()}; configured={TRUST_THRESHOLD}"
            ))

        # 10. NoTemporalMemory: Previous_Trust não deve dirigir New_Trust.
        # A propriedade operacional testável é New_Trust == Raw_Score.
        if {"Strategy", "Trust_After", "Raw_Score"}.issubset(log.columns):
            sub = log[log["Strategy"] == "Adaptive_NoTemporalMemory"].copy()
            if len(sub):
                diff = np.abs(
                    pd.to_numeric(sub["Trust_After"], errors="coerce") -
                    pd.to_numeric(sub["Raw_Score"], errors="coerce")
                )
                ok = bool((diff.fillna(np.inf) < 1e-10).all())
                checks.append(_validation_row(
                    "no_temporal_memory_new_trust_equals_raw_score",
                    ok,
                    f"max_abs_diff={diff.max()}"
                ))

        # 11. NoAdaptiveWeighting: clientes aceitos recebem peso 1; rejeitados 0.
        if {"Strategy", "Adaptive_Weight", "Client_Trusted"}.issubset(log.columns):
            sub = log[log["Strategy"] == "Adaptive_NoAdaptiveWeighting"].copy()
            if len(sub):
                expected_w = pd.to_numeric(sub["Client_Trusted"], errors="coerce").fillna(-1)
                observed_w = pd.to_numeric(sub["Adaptive_Weight"], errors="coerce")
                diff = np.abs(observed_w - expected_w)
                ok = bool((diff.fillna(np.inf) < 1e-10).all())
                checks.append(_validation_row(
                    "no_adaptive_weighting_uses_binary_weights",
                    ok,
                    f"max_abs_diff={diff.max()}"
                ))

        # 12. Full: memória temporal deve de fato produzir ao menos uma transição
        # com New_Trust diferente de Raw_Score, quando houver sinal temporal.
        if {"Strategy", "Trust_After", "Raw_Score"}.issubset(log.columns):
            sub = log[log["Strategy"] == "Adaptive_Ultimate"].copy()
            if len(sub):
                diff = np.abs(
                    pd.to_numeric(sub["Trust_After"], errors="coerce") -
                    pd.to_numeric(sub["Raw_Score"], errors="coerce")
                )
                observed_memory_effect = bool((diff.fillna(0) > 1e-10).any())
                checks.append(_validation_row(
                    "full_variant_exhibits_temporal_memory_effect",
                    observed_memory_effect,
                    f"max_abs_diff={diff.max()}",
                    severity="WARNING"
                ))

        # 13. Integridade por cliente/round: sem duplicata dentro de estratégia.
        dup_cols = [c for c in ["Strategy", "Seed", "Round", "Client"] if c in log.columns]
        if len(dup_cols) == 4:
            ndup = int(log.duplicated(dup_cols).sum())
            checks.append(_validation_row(
                "no_duplicate_client_round_rows",
                ndup == 0,
                f"duplicates={ndup}"
            ))
    else:
        checks.append(_validation_row(
            "adaptive_client_log_available",
            False,
            "adaptive_client_log ausente ou vazio"
        ))

    report = pd.DataFrame(checks)
    report["Status"] = np.where(report["Passed"], "PASS", "FAIL")
    return report


def assert_smoke_test_passed(validation_df, fail_fast=True):
    """
    Reprova o piloto apenas em checks ERROR.
    WARNINGs são mantidos no relatório para inspeção científica.
    """
    if validation_df is None or validation_df.empty:
        raise AssertionError("Relatório de smoke test vazio.")

    errors = validation_df[
        (validation_df["Severity"] == "ERROR") &
        (~validation_df["Passed"].astype(bool))
    ]

    if len(errors):
        msg = "Smoke test reprovado:\n" + "\n".join(
            f"- {r.Check}: {r.Details}" for r in errors.itertuples()
        )
        if fail_fast:
            raise AssertionError(msg)
        print(msg)
        return False

    print("Smoke test aprovado: nenhum erro estrutural/comportamental crítico.")
    return True


def save_smoke_test_validation(
    out_dir,
    raw_df,
    adaptive_client_log_df,
    expected_rounds=N_FEDERATED_ROUNDS,
):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    validation = validate_smoke_test_outputs(
        raw_df=raw_df,
        adaptive_client_log_df=adaptive_client_log_df,
        expected_rounds=expected_rounds,
        expected_strategies=STRATEGIES,
    )

    validation.to_csv(out_dir / "smoke_test_validation.csv", index=False)

    summary = {
        "Run_Version": RUN_VERSION,
        "Expected_Rounds": int(expected_rounds),
        "Expected_Strategies": int(len(STRATEGIES)),
        "Checks_Total": int(len(validation)),
        "Checks_Passed": int(validation["Passed"].sum()),
        "Checks_Failed": int((~validation["Passed"]).sum()),
        "Critical_Errors": int(
            ((validation["Severity"] == "ERROR") & (~validation["Passed"])).sum()
        ),
        "Warnings": int(
            ((validation["Severity"] == "WARNING") & (~validation["Passed"])).sum()
        ),
    }
    pd.DataFrame([summary]).to_csv(
        out_dir / "smoke_test_validation_summary.csv", index=False
    )

    assert_smoke_test_passed(validation, fail_fast=SMOKE_TEST_FAIL_FAST)
    return validation



# ==============================================================================
# AUDITORIA DE ABLAÇÃO RODADA A RODADA
# ==============================================================================

ABLATION_AUDIT_VARIANTS = {
    "Full": "Adaptive_Ultimate",
    "NoTemporalMemory": "Adaptive_NoTemporalMemory",
    "NoAdaptiveWeighting": "Adaptive_NoAdaptiveWeighting",
    "NoCircuitBreaker": "Adaptive_NoCircuitBreaker",
}


def build_ablation_round_audit(raw_df, client_log_df):
    """
    Consolida, por Dataset x Attack x Partition_Mode x Seed x Round,
    os sinais internos do Full e das três ablações de componente.
    O controle `Adaptive_FixedCircuitBreaker` é analisado separadamente em
    `circuit_breaker_comparison.csv`, evitando tratá-lo como ablação de componente.

    A auditoria permite observar:
      - efeito da memória: Full vs NoTemporalMemory;
      - efeito da ponderação: Full vs NoAdaptiveWeighting;
      - efeito do CB: Full vs NoCircuitBreaker;
      - consequência em F1/ASR e norma da atualização global, quando disponível.
    """
    if raw_df is None or len(raw_df) == 0:
        return pd.DataFrame()

    raw = raw_df.copy()
    logs = client_log_df.copy() if client_log_df is not None else pd.DataFrame()

    adaptive_strategies = set(ABLATION_AUDIT_VARIANTS.values())
    raw = raw[raw["Strategy"].isin(adaptive_strategies)].copy()
    if raw.empty:
        return pd.DataFrame()

    group_keys = [
        c for c in ["Dataset", "Attack", "Partition_Mode", "Seed", "Round"]
        if c in raw.columns
    ]

    result_cols = [
        c for c in [
            "Strategy", "F1", "ASR", "Circuit_Breaker", "Trusted_Clients",
            "Aggregate_Update_Norm", "Attack_Active", "Active_Attackers",
            "Attack_Pressure_Client_Fraction", "Legacy_Stage_Equivalent"
        ]
        if c in raw.columns
    ]

    # Uma linha por estratégia/round no dataframe global.
    global_round = (
        raw[group_keys + result_cols]
        .drop_duplicates(subset=group_keys + ["Strategy"])
        .copy()
    )

    # Mapeia nome de Strategy -> variante curta.
    reverse_variant = {v: k for k, v in ABLATION_AUDIT_VARIANTS.items()}
    global_round["Variant"] = global_round["Strategy"].map(reverse_variant)

    # Pivot das métricas globais.
    value_cols = [
        c for c in [
            "F1", "ASR", "Circuit_Breaker", "Trusted_Clients",
            "Aggregate_Update_Norm"
        ]
        if c in global_round.columns
    ]
    pivots = []
    for metric in value_cols:
        p = (
            global_round.pivot_table(
                index=group_keys,
                columns="Variant",
                values=metric,
                aggfunc="first"
            )
            .reset_index()
        )
        p.columns = [
            col if isinstance(col, str) else "_".join(str(x) for x in col if str(x))
            for col in p.columns
        ]
        ren = {
            v: f"{metric}_{v}"
            for v in ABLATION_AUDIT_VARIANTS.keys()
            if v in p.columns
        }
        p = p.rename(columns=ren)
        pivots.append(p)

    if pivots:
        audit = pivots[0]
        for p in pivots[1:]:
            audit = audit.merge(p, on=group_keys, how="outer")
    else:
        audit = raw[group_keys].drop_duplicates().copy()

    # Adiciona metadados do ataque uma única vez.
    meta_cols = [
        c for c in [
            "Attack_Active", "Active_Attackers",
            "Attack_Pressure_Client_Fraction", "Legacy_Stage_Equivalent"
        ]
        if c in global_round.columns
    ]
    if meta_cols:
        meta = (
            global_round[group_keys + meta_cols]
            .drop_duplicates(subset=group_keys)
        )
        audit = audit.merge(meta, on=group_keys, how="left")

    # Sinais internos por cliente.
    if not logs.empty and "Strategy" in logs.columns:
        logs = logs[logs["Strategy"].isin(adaptive_strategies)].copy()

        client_keys = [
            c for c in ["Dataset", "Attack", "Partition_Mode", "Seed", "Round", "Client"]
            if c in logs.columns
        ]

        internal_metrics = [
            c for c in [
                "Raw_Score", "Trust_Before", "Trust_After",
                "Adaptive_Weight", "Client_Trusted",
                "Client_Attack_Active", "Client_Is_Attacker"
            ]
            if c in logs.columns
        ]

        if client_keys and internal_metrics:
            logs["Variant"] = logs["Strategy"].map(reverse_variant)

            # Uma linha por cliente/round com colunas das quatro variantes.
            client_base = logs[client_keys].drop_duplicates().copy()

            for metric in internal_metrics:
                p = (
                    logs.pivot_table(
                        index=client_keys,
                        columns="Variant",
                        values=metric,
                        aggfunc="first"
                    )
                    .reset_index()
                )
                ren = {
                    v: f"{metric}_{v}"
                    for v in ABLATION_AUDIT_VARIANTS.keys()
                    if v in p.columns
                }
                p = p.rename(columns=ren)
                client_base = client_base.merge(p, on=client_keys, how="left")

            # Métricas contrafactuais explícitas.
            if {"Trust_After_Full", "Trust_After_NoTemporalMemory"}.issubset(client_base.columns):
                client_base["Delta_Trust_Memory"] = (
                    client_base["Trust_After_Full"]
                    - client_base["Trust_After_NoTemporalMemory"]
                )

            if {"Adaptive_Weight_Full", "Adaptive_Weight_NoAdaptiveWeighting"}.issubset(client_base.columns):
                client_base["Delta_Weight_Adaptive"] = (
                    client_base["Adaptive_Weight_Full"]
                    - client_base["Adaptive_Weight_NoAdaptiveWeighting"]
                )

            # Junta métricas globais de cada rodada a cada cliente.
            audit = client_base.merge(audit, on=[
                c for c in group_keys if c in client_base.columns
            ], how="left")

    # Efeitos globais marginais do componente removido.
    for metric in ["F1", "ASR", "Aggregate_Update_Norm"]:
        full_col = f"{metric}_Full"
        comparisons = {
            "Memory": f"{metric}_NoTemporalMemory",
            "Weighting": f"{metric}_NoAdaptiveWeighting",
            "CircuitBreaker": f"{metric}_NoCircuitBreaker",
        }
        if full_col in audit.columns:
            for comp_name, other_col in comparisons.items():
                if other_col in audit.columns:
                    audit[f"Delta_{metric}_{comp_name}"] = (
                        audit[full_col] - audit[other_col]
                    )

    sort_cols = [
        c for c in ["Dataset", "Attack", "Partition_Mode", "Seed", "Client", "Round"]
        if c in audit.columns
    ]
    if sort_cols:
        audit = audit.sort_values(sort_cols).reset_index(drop=True)

    return audit


def _save_ablation_plot(fig, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fig.savefig(path, dpi=220, bbox_inches="tight")
    plt.close(fig)


def generate_ablation_round_plots(audit_df, out_dir):
    """
    Gera figuras por seed:
      1) memória: RawScore/Trust Full vs NoTemporalMemory;
      2) ponderação: AdaptiveWeight Full vs NoAdaptiveWeighting;
      3) Circuit Breaker: TrustedClients e CB Full vs NoCircuitBreaker;
      4) desempenho: F1/ASR das quatro variantes Adaptive.
    """
    if audit_df is None or audit_df.empty:
        return []

    out_dir = Path(out_dir) / "ablation_round_plots"
    out_dir.mkdir(parents=True, exist_ok=True)
    created = []

    seed_values = sorted(audit_df["Seed"].dropna().unique()) if "Seed" in audit_df.columns else [None]

    for seed in seed_values:
        sdf = audit_df[audit_df["Seed"] == seed].copy() if seed is not None else audit_df.copy()

        # -------- 1. Memória temporal --------
        if "Client" in sdf.columns and {
            "Round", "Trust_After_Full", "Trust_After_NoTemporalMemory"
        }.issubset(sdf.columns):
            fig, ax = plt.subplots(figsize=(12, 6))
            for client, cdf in sdf.groupby("Client"):
                cdf = cdf.sort_values("Round")
                ax.plot(cdf["Round"], cdf["Trust_After_Full"], label=f"{client} Full")
                ax.plot(
                    cdf["Round"],
                    cdf["Trust_After_NoTemporalMemory"],
                    linestyle="--",
                    label=f"{client} NoMemory"
                )
            ax.axhline(TRUST_THRESHOLD, linestyle=":", label=f"Trust threshold={TRUST_THRESHOLD}")
            ax.set_title(f"Ablation — Temporal Memory | Seed {seed}")
            ax.set_xlabel("Federated round")
            ax.set_ylabel("Trust")
            ax.legend(ncol=2, fontsize=8)
            p = out_dir / f"memory_full_vs_no_memory_seed_{seed}.png"
            _save_ablation_plot(fig, p)
            created.append(str(p))

        # -------- 2. Ponderação adaptativa --------
        if "Client" in sdf.columns and {
            "Round", "Adaptive_Weight_Full", "Adaptive_Weight_NoAdaptiveWeighting"
        }.issubset(sdf.columns):
            fig, ax = plt.subplots(figsize=(12, 6))
            for client, cdf in sdf.groupby("Client"):
                cdf = cdf.sort_values("Round")
                ax.plot(cdf["Round"], cdf["Adaptive_Weight_Full"], label=f"{client} Full")
                ax.plot(
                    cdf["Round"],
                    cdf["Adaptive_Weight_NoAdaptiveWeighting"],
                    linestyle="--",
                    label=f"{client} NoWeight"
                )
            ax.set_title(f"Ablation — Adaptive Weighting | Seed {seed}")
            ax.set_xlabel("Federated round")
            ax.set_ylabel("Aggregation weight")
            ax.legend(ncol=2, fontsize=8)
            p = out_dir / f"weight_full_vs_no_weight_seed_{seed}.png"
            _save_ablation_plot(fig, p)
            created.append(str(p))

        # -------- 3. Circuit Breaker --------
        # Usa uma linha por round para evitar repetição por cliente.
        round_df = sdf.drop_duplicates(subset=["Round"]).sort_values("Round")
        if {
            "Round", "Trusted_Clients_Full", "Circuit_Breaker_Full",
            "Circuit_Breaker_NoCircuitBreaker"
        }.issubset(round_df.columns):
            fig, ax = plt.subplots(figsize=(12, 6))
            ax.plot(
                round_df["Round"],
                round_df["Trusted_Clients_Full"],
                label="Trusted clients — Full"
            )
            # majority inferida do número máximo de clientes observado na auditoria
            if "Client" in sdf.columns:
                n_clients = sdf["Client"].nunique()
                majority = (n_clients // 2) + 1
                ax.axhline(majority, linestyle=":", label=f"Majority threshold={majority}")

            cb_rounds = round_df.loc[
                pd.to_numeric(round_df["Circuit_Breaker_Full"], errors="coerce").fillna(0) > 0,
                "Round"
            ]
            for r in cb_rounds:
                ax.axvline(r, alpha=0.15)

            ax.set_title(f"Ablation — Circuit Breaker | Seed {seed}")
            ax.set_xlabel("Federated round")
            ax.set_ylabel("Trusted clients")
            ax.legend()
            p = out_dir / f"circuit_breaker_full_vs_no_cb_seed_{seed}.png"
            _save_ablation_plot(fig, p)
            created.append(str(p))

        # -------- 4. F1 e ASR --------
        if "Round" in round_df.columns:
            f1_cols = [
                c for c in [
                    "F1_Full", "F1_NoTemporalMemory",
                    "F1_NoAdaptiveWeighting", "F1_NoCircuitBreaker"
                ]
                if c in round_df.columns
            ]
            if f1_cols:
                fig, ax = plt.subplots(figsize=(12, 6))
                for c in f1_cols:
                    ax.plot(round_df["Round"], round_df[c], label=c.replace("F1_", ""))
                ax.set_title(f"Ablation — F1 by round | Seed {seed}")
                ax.set_xlabel("Federated round")
                ax.set_ylabel("F1")
                ax.legend()
                p = out_dir / f"f1_ablation_by_round_seed_{seed}.png"
                _save_ablation_plot(fig, p)
                created.append(str(p))

            asr_cols = [
                c for c in [
                    "ASR_Full", "ASR_NoTemporalMemory",
                    "ASR_NoAdaptiveWeighting", "ASR_NoCircuitBreaker"
                ]
                if c in round_df.columns
            ]
            if asr_cols:
                fig, ax = plt.subplots(figsize=(12, 6))
                for c in asr_cols:
                    ax.plot(round_df["Round"], round_df[c], label=c.replace("ASR_", ""))
                ax.set_title(f"Ablation — ASR by round | Seed {seed}")
                ax.set_xlabel("Federated round")
                ax.set_ylabel("ASR")
                ax.legend()
                p = out_dir / f"asr_ablation_by_round_seed_{seed}.png"
                _save_ablation_plot(fig, p)
                created.append(str(p))

    return created


def save_ablation_round_audit(raw_df, client_log_df, out_dir):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    audit_df = build_ablation_round_audit(raw_df, client_log_df)
    audit_path = out_dir / "ablation_round_audit.csv"
    audit_df.to_csv(audit_path, index=False)

    created_plots = generate_ablation_round_plots(audit_df, out_dir)

    manifest = pd.DataFrame({
        "Artifact": ["ablation_round_audit.csv"] + [Path(p).name for p in created_plots],
        "Type": ["csv"] + ["plot"] * len(created_plots),
    })
    manifest.to_csv(out_dir / "ablation_round_audit_manifest.csv", index=False)

    return audit_df, created_plots



# ==============================================================================
# CUMULATIVE STRESS TEST — MÉTRICAS ESPECÍFICAS
# ==============================================================================

def save_cumulative_stress_analysis(all_raw):
    """Gera métricas de sensibilidade por seed e resumo agregado.

    Ponto de ruptura é descritivo, não um critério de seleção do perfil,
    e é calculado somente em rounds com Attack_Active=1:
      - primeiro round ativo com ASR > 0.10
      - primeiro round ativo com F1 < 0.95
    """
    df = all_raw.copy()
    if df.empty:
        return pd.DataFrame(), pd.DataFrame()

    rows = []
    group_cols = ["Dataset", "Partition_Mode", "Cumulative_Profile", "Seed", "Strategy"]
    for keys, g in df.groupby(group_cols):
        g = g.sort_values("Round")
        dataset, partition_mode, profile, seed, strategy = keys
        pre = g[g["Attack_Active"] == 0]
        active = g[g["Attack_Active"] == 1]
        critical = g[g["Legacy_Stage_Equivalent"].between(12, 16)]
        final = g.iloc[-1]

        baseline_f1 = float(pre["F1"].mean()) if len(pre) else np.nan
        final_f1 = float(final["F1"])

        # Breakpoints devem representar ruptura causada durante o ataque,
        # não instabilidade de warm-up anterior ao round de ativação.
        asr_break = active[active["ASR"] > 0.10]
        f1_break = active[active["F1"] < 0.95]

        rows.append({
            "Dataset": dataset,
            "Partition_Mode": partition_mode,
            "Cumulative_Profile": profile,
            "Seed": int(seed),
            "Strategy": strategy,
            "Baseline_F1_PreAttack": baseline_f1,
            "F1_Final": final_f1,
            "Delta_F1_PreAttack_to_Final": baseline_f1 - final_f1 if np.isfinite(baseline_f1) else np.nan,
            "ASR_Attack_Active_Mean": float(active["ASR"].mean()) if len(active) else np.nan,
            "ASR_Critical_S12_S16_Mean": float(critical["ASR"].mean()) if len(critical) else np.nan,
            "F1_Critical_S12_S16_Mean": float(critical["F1"].mean()) if len(critical) else np.nan,
            "ASR_Peak": float(active["ASR"].max()) if len(active) else np.nan,
            "ASR_Final": float(final["ASR"]),
            "Final_Epsilon": float(final["Cumulative_Epsilon"]),
            "First_Round_ASR_GT_0_10": int(asr_break["Round"].iloc[0]) if len(asr_break) else np.nan,
            "First_Round_F1_LT_0_95": int(f1_break["Round"].iloc[0]) if len(f1_break) else np.nan,
        })

    by_seed = pd.DataFrame(rows)
    by_seed.to_csv(OUTPUT_ROOT / "cumulative_stress_breakpoints_by_seed.csv", index=False)

    summary = (
        by_seed
        .groupby(["Dataset", "Partition_Mode", "Cumulative_Profile", "Strategy"], as_index=False)
        .agg(
            N_Seeds=("Seed", "nunique"),
            Baseline_F1_PreAttack_Mean=("Baseline_F1_PreAttack", "mean"),
            F1_Final_Mean=("F1_Final", "mean"),
            Delta_F1_Mean=("Delta_F1_PreAttack_to_Final", "mean"),
            ASR_Attack_Active_Mean=("ASR_Attack_Active_Mean", "mean"),
            ASR_Critical_S12_S16_Mean=("ASR_Critical_S12_S16_Mean", "mean"),
            F1_Critical_S12_S16_Mean=("F1_Critical_S12_S16_Mean", "mean"),
            ASR_Peak_Mean=("ASR_Peak", "mean"),
            ASR_Final_Mean=("ASR_Final", "mean"),
            Final_Epsilon=("Final_Epsilon", "max"),
            First_Round_ASR_GT_0_10_Mean=("First_Round_ASR_GT_0_10", "mean"),
            First_Round_F1_LT_0_95_Mean=("First_Round_F1_LT_0_95", "mean"),
        )
    )
    summary.to_csv(OUTPUT_ROOT / "cumulative_stress_summary.csv", index=False)
    return by_seed, summary


def _holm_adjust(p_values):
    """Correção de Holm para uma família de p-valores."""
    p = np.asarray(p_values, dtype=float)
    out = np.full(len(p), np.nan, dtype=float)
    valid = np.where(np.isfinite(p))[0]
    if len(valid) == 0:
        return out
    order = valid[np.argsort(p[valid])]
    m = len(order)
    running = 0.0
    for rank, idx in enumerate(order):
        adjusted = (m - rank) * p[idx]
        running = max(running, adjusted)
        out[idx] = min(1.0, running)
    return out


def save_cumulative_base_vs_extreme_statistics(by_seed, temporal_summary_all):
    """Comparação pareada Base vs Extreme usando Seed como unidade experimental.

    Gera duas famílias de métricas:
      1) impacto global: ASR/F1 crítico em S12-S16;
      2) resposta temporal: recall, delay e rejeição honesta do mecanismo adaptativo.
    Não trata rounds como réplicas independentes.
    """
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

    # Métricas globais por seed/estratégia já produzidas pelo stress analyzer.
    impact = by_seed[[
        "Dataset", "Partition_Mode", "Cumulative_Profile", "Seed", "Strategy",
        "ASR_Critical_S12_S16_Mean", "F1_Critical_S12_S16_Mean",
        "ASR_Attack_Active_Mean", "ASR_Peak", "ASR_Final", "F1_Final"
    ]].copy()

    temporal = pd.DataFrame()
    if temporal_summary_all is not None and not temporal_summary_all.empty:
        keep = [
            "Dataset", "Partition_Mode", "Cumulative_Profile", "Seed", "Strategy",
            "Malicious_Detection_Recall", "Detection_Delay_Mean", "Honest_Rejection_Rate",
            "Malicious_Detection_Precision", "CB_Trigger_Count", "CB_Active_Rounds"
        ]
        temporal = temporal_summary_all[[c for c in keep if c in temporal_summary_all.columns]].copy()

    metrics = {
        "ASR_Critical_S12_S16_Mean": "lower",
        "F1_Critical_S12_S16_Mean": "higher",
        "Malicious_Detection_Recall": "higher",
        "Detection_Delay_Mean": "lower",
        "Honest_Rejection_Rate": "lower",
    }

    merged = impact.copy()
    if not temporal.empty:
        join_cols = ["Dataset", "Partition_Mode", "Cumulative_Profile", "Seed", "Strategy"]
        merged = merged.merge(temporal, on=join_cols, how="left")
    merged.to_csv(OUTPUT_ROOT / "cumulative_final_metrics_by_seed.csv", index=False)

    stat_rows = []
    for (dataset, partition, strategy), g in merged.groupby(["Dataset", "Partition_Mode", "Strategy"]):
        base = g[g["Cumulative_Profile"] == "Base"].set_index("Seed")
        extreme = g[g["Cumulative_Profile"] == "Extreme"].set_index("Seed")
        common_seeds = sorted(set(base.index).intersection(extreme.index))
        if not common_seeds:
            continue
        for metric, direction in metrics.items():
            if metric not in g.columns:
                continue
            x = pd.to_numeric(base.loc[common_seeds, metric], errors="coerce")
            y = pd.to_numeric(extreme.loc[common_seeds, metric], errors="coerce")
            mask = x.notna() & y.notna()
            x = x[mask].to_numpy(dtype=float)
            y = y[mask].to_numpy(dtype=float)
            n = len(x)
            if n == 0:
                continue
            d = y - x  # Extreme - Base
            mean_diff = float(np.mean(d))
            if n >= 2 and np.std(d, ddof=1) > 0:
                se = np.std(d, ddof=1) / np.sqrt(n)
                tcrit = stats.t.ppf(0.975, df=n-1)
                ci_low = mean_diff - tcrit * se
                ci_high = mean_diff + tcrit * se
                cohen_dz = mean_diff / np.std(d, ddof=1)
            else:
                ci_low = ci_high = cohen_dz = np.nan
            try:
                if n >= 2 and np.any(np.abs(d) > 0):
                    wil = stats.wilcoxon(y, x, alternative="two-sided", zero_method="wilcox")
                    p_raw = float(wil.pvalue)
                    w_stat = float(wil.statistic)
                else:
                    p_raw = 1.0
                    w_stat = 0.0
            except Exception:
                p_raw = np.nan
                w_stat = np.nan
            stat_rows.append({
                "Dataset": dataset,
                "Partition_Mode": partition,
                "Strategy": strategy,
                "Metric": metric,
                "Preferred_Direction": direction,
                "N_Paired_Seeds": n,
                "Base_Mean": float(np.mean(x)),
                "Extreme_Mean": float(np.mean(y)),
                "Extreme_minus_Base": mean_diff,
                "Paired_Diff_CI95_Low": ci_low,
                "Paired_Diff_CI95_High": ci_high,
                "Wilcoxon_W": w_stat,
                "Wilcoxon_p_raw": p_raw,
                "Cohen_dz": cohen_dz,
            })

    stats_df = pd.DataFrame(stat_rows)
    if not stats_df.empty:
        stats_df["Wilcoxon_p_Holm"] = np.nan
        # Família de 5 métricas dentro de cada partição x estratégia.
        for _, idx in stats_df.groupby(["Dataset", "Partition_Mode", "Strategy"]).groups.items():
            idx = list(idx)
            stats_df.loc[idx, "Wilcoxon_p_Holm"] = _holm_adjust(stats_df.loc[idx, "Wilcoxon_p_raw"].values)
        stats_df["Significant_Holm_0_05"] = stats_df["Wilcoxon_p_Holm"] < 0.05
        stats_df.to_csv(OUTPUT_ROOT / "cumulative_base_vs_extreme_statistics.csv", index=False)
        stats_df[stats_df["Strategy"] == "Adaptive_Ultimate"].to_csv(
            OUTPUT_ROOT / "cumulative_base_vs_extreme_thesis_focus_adaptive.csv", index=False
        )
    return merged, stats_df

def main():
    effective_partitions = CLIENT_PARTITION_MODES

    print("CUMULATIVE FINAL BASE vs EXTREME — CSAI-4-IoT | PÓS-BANCA 2026")
    print("Perfis:", CUMULATIVE_PROFILES_TO_RUN)
    print(f"Modo: {CUMULATIVE_STRESS_MODE} | Seeds: {RUN_SEEDS}")
    print(f"Rounds: {N_FEDERATED_ROUNDS} | tau={TRUST_THRESHOLD:.2f}")
    print("Partições:", effective_partitions)
    print("Estratégias:", STRATEGIES)
    print("Controle Base: epsilon=0.20 constante durante ataque ativo")
    print("Comparação final: Base epsilon=0.20 constante vs Extreme 0.20 -> 4.00")

    if IN_COLAB and COLAB_DRIVE_MODE:
        print("PROJECT_ROOT :", PROJECT_ROOT)
        print("DATASET_DIR  :", DATASET_DIR)
        print("OUTPUT_ROOT  :", OUTPUT_ROOT)

        if not DATASET_DIR.exists():
            raise FileNotFoundError(
                "Diretório oficial dos 9 dispositivos não encontrado:\n"
                f"{DATASET_DIR}"
            )

        datasets = []
        missing_files = []
        for filename in EXPECTED_DEVICE_FILES:
            csv_path = DATASET_DIR / filename
            if not csv_path.exists():
                missing_files.append(filename)
                continue
            df = pd.read_csv(csv_path)
            dataset_name = detect_dataset_name(csv_path.name, df)
            clean_df, target_col = preprocess_dataframe(df)
            datasets.append({
                "filename": csv_path.name,
                "dataset_name": dataset_name,
                "df": clean_df,
                "target_col": target_col,
            })

        if missing_files:
            raise FileNotFoundError(
                "Cumulative Stress Test abortado: faltam datasets-fonte esperados:\n- "
                + "\n- ".join(missing_files)
            )
        if len(datasets) != 9:
            raise RuntimeError(f"Esperados 9 datasets; carregados {len(datasets)}.")
    elif IN_COLAB:
        raise RuntimeError("Notebook preparado para COLAB_DRIVE_MODE=True.")
    else:
        datasets = []
        for filename in EXPECTED_DEVICE_FILES:
            csv_path = DATASET_DIR / filename
            if not csv_path.exists():
                continue
            df = pd.read_csv(csv_path)
            dataset_name = detect_dataset_name(csv_path.name, df)
            clean_df, target_col = preprocess_dataframe(df)
            datasets.append({
                "filename": csv_path.name,
                "dataset_name": dataset_name,
                "df": clean_df,
                "target_col": target_col,
            })
        if len(datasets) != 9:
            raise RuntimeError(
                f"Execução local: esperados 9 datasets em {DATASET_DIR}; encontrados {len(datasets)}."
            )

    # Mesma harmonização usada na campanha final.
    combined_df, harmonization_manifest = harmonize_and_combine_datasets(
        datasets, seed=INITIAL_SEED
    )
    harmonization_manifest.to_csv(OUTPUT_ROOT / "harmonization_manifest.csv", index=False)
    if combined_df is None:
        raise RuntimeError("Falha ao criar o dataset mixed-device harmonizado.")

    dataset_name = COMBINED_DATASET_NAME
    df = combined_df
    target_col = "class"

    print("\nDataset combinado validado:")
    print(f"  {dataset_name} | linhas={len(df)} | features={df.shape[1]-1}")

    all_results = []
    execution_rows = []
    temporal_summary_frames = []

    for profile in CUMULATIVE_PROFILES_TO_RUN:
        print("\n" + "=" * 90)
        print(f"CUMULATIVE PROFILE: {profile}")
        print(CUMULATIVE_STRESS_PROFILES[profile]["description"])
        print("=" * 90)

        for partition_mode in effective_partitions:
            partition_start = time.perf_counter()
            attack_results = []
            attack_client_logs = []

            for seed in RUN_SEEDS:
                seed_results, seed_client_logs = run_single_seed(
                    df=df,
                    target_col=target_col,
                    seed=seed,
                    dataset_name=dataset_name,
                    attack_name="Cumulative",
                    partition_mode=partition_mode,
                    cumulative_profile=profile,
                )
                attack_results.extend(seed_results)
                attack_client_logs.extend(seed_client_logs)

            partition_total_time = time.perf_counter() - partition_start
            attack_df = pd.DataFrame(attack_results)
            client_log_df = pd.DataFrame(attack_client_logs)

            out_dir = (
                OUTPUT_ROOT
                / safe_slug(dataset_name)
                / "Cumulative_Stress"
                / profile
                / safe_slug(partition_mode)
            )

            save_outputs(attack_df, out_dir)
            if not client_log_df.empty:
                client_log_df.to_csv(out_dir / "adaptive_client_log.csv", index=False)
                temporal_episodes, cb_events, temporal_summary = save_temporal_defense_analysis(client_log_df, out_dir)
                if not temporal_summary.empty:
                    temporal_summary = temporal_summary.copy()
                    temporal_summary["Cumulative_Profile"] = profile
                    temporal_summary_frames.append(temporal_summary)
                ablation_audit_df, ablation_plots = save_ablation_round_audit(
                    raw_df=attack_df,
                    client_log_df=client_log_df,
                    out_dir=out_dir,
                )
                print(
                    f"  Temporal: {len(temporal_episodes)} episódios | {len(cb_events)} eventos CB | "
                    f"Ablation audit: {len(ablation_audit_df)} linhas"
                )

            execution_rows.append({
                "Dataset": dataset_name,
                "Attack": "Cumulative",
                "Cumulative_Profile": profile,
                "Partition_Mode": partition_mode,
                "Dirichlet_Alpha": NONIID_DIRICHLET_ALPHA if partition_mode == "NonIID_Dirichlet" else np.nan,
                "Seeds": len(RUN_SEEDS),
                "Seed_Start": min(RUN_SEEDS),
                "Seed_End": max(RUN_SEEDS),
                "Trust_Threshold": TRUST_THRESHOLD,
                "Strategies": len(STRATEGIES),
                "Rounds": N_FEDERATED_ROUNDS,
                "Rows": len(df),
                "Features": df.shape[1] - 1,
                "Epsilon_Start_Active": float(
                    CUMULATIVE_STRESS_PROFILE[
                        (CUMULATIVE_STRESS_PROFILE["Profile"] == profile) &
                        (CUMULATIVE_STRESS_PROFILE["Attack_Active"] == 1)
                    ]["Epsilon"].iloc[0]
                ),
                "Epsilon_Final": float(cumulative_epsilon_for_round(99, profile)),
                "Persistent_Attackers": 1,
                "Total_Execution_Time_Seconds": partition_total_time,
                "Total_Execution_Time_Minutes": partition_total_time / 60.0,
            })

            pd.DataFrame([execution_rows[-1]]).to_csv(out_dir / "execution_report.csv", index=False)
            all_results.append(attack_df)
            print(
                f"Concluído: {profile} — {partition_mode} | "
                f"{partition_total_time/60.0:.2f} min"
            )

    if not all_results:
        raise RuntimeError("Nenhum resultado gerado.")

    all_raw = pd.concat(all_results, ignore_index=True)
    all_raw.to_csv(OUTPUT_ROOT / "all_raw_results.csv", index=False)
    _, all_final, diagnostic = save_global_outputs(all_raw)

    by_seed, stress_summary = save_cumulative_stress_analysis(all_raw)
    temporal_summary_all = (
        pd.concat(temporal_summary_frames, ignore_index=True)
        if temporal_summary_frames else pd.DataFrame()
    )
    if not temporal_summary_all.empty:
        temporal_summary_all.to_csv(OUTPUT_ROOT / "cumulative_temporal_metrics_by_seed.csv", index=False)
    final_metrics_by_seed, profile_stats = save_cumulative_base_vs_extreme_statistics(
        by_seed, temporal_summary_all
    )
    pd.DataFrame(execution_rows).to_csv(OUTPUT_ROOT / "execution_report_all_profiles.csv", index=False)

    if len(RUN_SEEDS) >= 2:
        statistical_outputs = save_statistical_analysis(all_raw)
        print(
            f"\nEstatística: Friedman={len(statistical_outputs['friedman'])} | "
            f"Wilcoxon-Holm={len(statistical_outputs['pairwise'])}"
        )
    else:
        print("\nPiloto com 1 seed: inferência estatística intencionalmente não executada.")

    print("\nResumo específico do Cumulative Stress Test:")
    display(
        stress_summary.sort_values(
            ["Partition_Mode", "Cumulative_Profile", "ASR_Critical_S12_S16_Mean", "F1_Critical_S12_S16_Mean"],
            ascending=[True, True, True, False],
        )
    )

    completion = pd.DataFrame([{
        "Run": RUN_VERSION,
        "Mode": CUMULATIVE_STRESS_MODE,
        "Seed_Start": min(RUN_SEEDS),
        "Seed_End": max(RUN_SEEDS),
        "Seeds": len(RUN_SEEDS),
        "Calibration_Seed_Excluded": CALIBRATION_SEED,
        "Trust_Threshold": TRUST_THRESHOLD,
        "Rounds": N_FEDERATED_ROUNDS,
        "Attack": "Cumulative",
        "Profiles": ",".join(CUMULATIVE_PROFILES_TO_RUN),
        "Partitions": ",".join(effective_partitions),
        "Strategies": len(STRATEGIES),
        "Dataset": dataset_name,
        "Status": "COMPLETED",
    }])
    completion.to_csv(OUTPUT_ROOT / "RUN_COMPLETED.csv", index=False)

    persistent_zip = OUTPUT_ROOT.parent / f"{COLAB_RESULTS_DIRNAME}.zip"
    if persistent_zip.exists():
        persistent_zip.unlink()
    shutil.make_archive(str(persistent_zip.with_suffix("")), "zip", OUTPUT_ROOT)
    print("\nZIP persistente salvo em:", persistent_zip)


if __name__ == "__main__":
    main()


Mounted at /content/drive

Bootstrap de caminhos validado:
  DRIVE_ROOT   : /content/drive/MyDrive
  PROJECT_ROOT : /content/drive/MyDrive/00_Tese_Benchmark/01_Benchmark_Pos_Banca_2026
  DATASET_DIR  : /content/drive/MyDrive/00_Tese_Benchmark/01_Benchmark_Pos_Banca_2026/02_DADOS_HARMONIZADOS/04_NBAIOT_DISPOSITIVOS_INDIVIDUAIS_BALANCEADOS/Seed_42
  OUTPUT_ROOT  : /content/drive/MyDrive/00_Tese_Benchmark/01_Benchmark_Pos_Banca_2026/09_RESULTADOS_FINAIS_TESE/08_CUMULATIVE_FINAL_BASE_VS_EXTREME_100R_SEEDS43_52

Perfil de pressão adversarial (S0..S16):
 Legacy_Stage  Pressure_Fraction  Compromised_Clients  OnOff_Active
            0                0.0                    0             0
            1                0.1                    1             1
            2                0.1                    1             0
            3                0.1                    1             1
            4                0.1                    1             0
            5                0.3      

,Dataset,Partition_Mode,Cumulative_Profile,Strategy,N_Seeds,Baseline_F1_PreAttack_Mean,F1_Final_Mean,Delta_F1_Mean,ASR_Attack_Active_Mean,ASR_Critical_S12_S16_Mean,F1_Critical_S12_S16_Mean,ASR_Peak_Mean,ASR_Final_Mean,Final_Epsilon,First_Round_ASR_GT_0_10_Mean,First_Round_F1_LT_0_95_Mean
1,NBaIoT_MixedDevice_9Sources_Balanced,IID,Base,Adaptive_NoAdaptiveWeighting,10,0.993214,0.997944,-0.004731,0.001335,0.001316,0.997924,0.001889,0.001444,0.2,NaN,NaN
6,NBaIoT_MixedDevice_9Sources_Balanced,IID,Base,FLTrust,10,0.993230,0.997944,-0.004715,0.001376,0.001418,0.997916,0.001889,0.001556,0.2,NaN,NaN
0,NBaIoT_MixedDevice_9Sources_Balanced,IID,Base,Adaptive_FixedCircuitBreaker,10,0.993222,0.997889,-0.004667,0.001351,0.001431,0.997913,0.001889,0.001556,0.2,NaN,NaN
2,NBaIoT_MixedDevice_9Sources_Balanced,IID,Base,Adaptive_NoCircuitBreaker,10,0.993222,0.997889,-0.004667,0.001351,0.001431,0.997896,0.001889,0.001556,0.2,NaN,NaN
3,NBaIoT_MixedDevice_9Sources_Balanced,IID,Base,Adaptive_NoTemporalMemory,10,0.993230,0.997889,-0.004659,0.001367,0.001431,0.997880,0.001889,0.001556,0.2,NaN,NaN
4,NBaIoT_MixedDevice_9Sources_Balanced,IID,Base,Adaptive_Ultimate,10,0.993222,0.997833,-0.004612,0.001351,0.001431,0.997858,0.001889,0.001556,0.2,NaN,NaN
7,NBaIoT_MixedDevice_9Sources_Balanced,IID,Base,FedAvg,10,0.989176,0.997944,-0.008769,0.001435,0.001449,0.997782,0.002000,0.001333,0.2,NaN,NaN
5,NBaIoT_MixedDevice_9Sources_Balanced,IID,Base,FLAME,10,0.989196,0.997722,-0.008526,0.001453,0.001613,0.997631,0.001889,0.001556,0.2,NaN,NaN
8,NBaIoT_MixedDevice_9Sources_Balanced,IID,Base,Krum,10,0.989709,0.997000,-0.007291,0.001602,0.001773,0.997124,0.002667,0.001889,0.2,NaN,NaN
10,NBaIoT_MixedDevice_9Sources_Balanced,IID,Extreme,Adaptive_NoAdaptiveWeighting,10,0.993214,0.997944,-0.004731,0.001154,0.001116,0.997853,0.001889,0.001222,4.0,NaN,NaN



ZIP persistente salvo em: /content/drive/MyDrive/00_Tese_Benchmark/01_Benchmark_Pos_Banca_2026/09_RESULTADOS_FINAIS_TESE/08_CUMULATIVE_FINAL_BASE_VS_EXTREME_100R_SEEDS43_52.zip
